# ROP Stage Classification Workflow

This notebook implements the final workflow for four ROP classification scenarios using the Zhao2024 dataset:

- S1: raw RGB fundus images.
- S2: conservative Rahim-inspired image enhancement.
- S3: soft Almeida vesselness map only.
- S4: vessel-guided enhanced RGB image.

The notebook is optimized for local execution with `uv` and for Kaggle execution with `/kaggle/input` and `/kaggle/working`. Deterministic preprocessing is cached; random training augmentation is not cached.

## Environment Notes

Local setup with `uv`:

```bash
uv sync
uv run python -m ipykernel install --user --name rop-stage-classification --display-name "ROP Stage Classification"
```

Kaggle usually already has the core packages. If a package is missing on Kaggle, install only the missing package in a separate setup cell, for example:

```python
%pip install -q torchmetrics albumentations scikit-image
```

In [ ]:
from pathlib import Path
import os
import sys
import csv
import json
import time
import math
import random
import warnings
from dataclasses import dataclass, asdict, replace
from typing import Dict, Iterable, List, Literal, Tuple, Optional

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle/input').exists()
PROJECT_ROOT = Path.cwd()
INPUT_ROOT = Path('/kaggle/input') if IS_KAGGLE else PROJECT_ROOT / 'data'
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT
OUTPUT_DIR = WORK_DIR / 'output'
OUTPUT_DEBUG_ROOT = OUTPUT_DIR / '00_debug_baseline'
OUTPUT_PREPROCESSING_DIR = OUTPUT_DIR / '01_preprocessing_cache'
OUTPUT_TRAINING_RUNS_DIR = OUTPUT_DIR / '02_training_runs'
OUTPUT_EVALUATION_DIR = OUTPUT_DIR / '03_evaluation'
OUTPUT_FIGURES_DIR = OUTPUT_DIR / '04_figures'
OUTPUT_SPLITS_DIR = OUTPUT_PREPROCESSING_DIR / 'splits'
OUTPUT_PROCESSED_DIR = OUTPUT_PREPROCESSING_DIR / 'processed'
OUTPUT_MODELS_DIR = OUTPUT_TRAINING_RUNS_DIR / 'models'
OUTPUT_RESULTS_DIR = OUTPUT_EVALUATION_DIR / 'results'
OUTPUT_DEBUG_BASELINE_DIR = OUTPUT_DEBUG_ROOT / 'normalization_residual_tuned'

# Optional manual dataset roots. Add Kaggle-mounted dataset paths here if auto-detection fails.
USER_DATA_ROOTS = [
    Path('/kaggle/input/datasets/ghiffaribravia/rop-stage-classification-datasets'),
] if IS_KAGGLE else []

os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR / '.matplotlib'))

for directory in [
    OUTPUT_DIR,
    OUTPUT_DIR / '.matplotlib',
    OUTPUT_DEBUG_ROOT,
    OUTPUT_DEBUG_BASELINE_DIR,
    OUTPUT_PREPROCESSING_DIR,
    OUTPUT_SPLITS_DIR,
    OUTPUT_PROCESSED_DIR,
    OUTPUT_TRAINING_RUNS_DIR,
    OUTPUT_MODELS_DIR,
    OUTPUT_EVALUATION_DIR,
    OUTPUT_RESULTS_DIR,
    OUTPUT_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f'IS_KAGGLE  : {IS_KAGGLE}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'INPUT_ROOT  : {INPUT_ROOT}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')
print('USER_DATA_ROOTS:')
for root in USER_DATA_ROOTS:
    print(f'  - {root} exists={root.exists()}')

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

from scipy import ndimage as ndi
from skimage.feature import hessian_matrix, hessian_matrix_eigvals
from skimage.filters import frangi, meijering, sato, threshold_triangle, threshold_otsu
from skimage.morphology import (
    binary_closing,
    binary_opening,
    disk,
    remove_small_holes,
    remove_small_objects,
    skeletonize,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models

try:
    import albumentations as A
except Exception as exc:
    A = None
    print('Albumentations is unavailable. Install it or use the fallback transforms. Error:', exc)

sns.set_theme(style='whitegrid')

In [ ]:
@dataclass
class CFG:
    seed: int = 42
    image_size: int = 224
    class_names: Tuple[str, ...] = ('Normal', 'Stage1', 'Stage2', 'Stage3')
    excluded_classes: Tuple[str, ...] = ('laser scars',)
    train_ratio: float = 0.8
    val_ratio: float = 0.1
    test_ratio: float = 0.1
    batch_size: int = 64
    batch_size_resnet50: int = 64
    epochs: int = 40
    patience: int = 8
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    num_workers: int = 2
    use_amp: bool = True
    use_multi_gpu: bool = True
    force_rebuild_cache: bool = False
    run_preprocessing: bool = False
    run_segmentation_eval: bool = False
    run_training: bool = False
    run_resnet50_baseline: bool = False
    debug_mode: bool = True
    debug_tile_size: int = 150
    debug_output_dir: str = str(OUTPUT_DEBUG_BASELINE_DIR)
    # Fixed baseline: output/00_debug_baseline/normalization_residual_tuned.
    vessel_process_max_side: int = 768
    vesselness_mode: str = 'almeida_paper'
    vessel_threshold_method: str = 'percentile'
    vessel_triangle_nbins: int = 4
    vessel_min_size: int = 50
    vessel_hole_size: int = 0
    vessel_fov_erode_px: int = 12
    vessel_target_density: float = 0.10
    vessel_morphology: str = 'none'
    vessel_od_suppression: bool = False
    vessel_od_soft_penalty: float = 0.35
    vessel_final_skeletonize: bool = False
    vessel_skeleton_min_area: int = 20
    vessel_skeleton_dilate_iter: int = 1
    vessel_auto_binary_selection: bool = False
    vessel_top_hat_weight: float = 0.05
    vessel_matched_weight: float = 0.25
    vessel_frangi_weight: float = 0.20
    vessel_jerman_weight: float = 0.50
    vessel_guide_floor: float = 0.75
    vessel_guide_dilate_iter: int = 2
    vessel_guide_blur_sigma: float = 3.0

cfg = CFG()
SCENARIOS = ('S1_raw', 'S2_enhanced', 'S3_vessel_mask', 'S4_vessel_guided')
CLASS_TO_ID = {name: idx for idx, name in enumerate(cfg.class_names)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}

print(json.dumps(asdict(cfg), indent=2))

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(cfg.seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_COUNT = torch.cuda.device_count()
USE_AMP = bool(cfg.use_amp and DEVICE.type == 'cuda')
USE_MULTI_GPU = bool(cfg.use_multi_gpu and GPU_COUNT > 1)

print(f'Device: {DEVICE}')
print(f'CUDA devices: {GPU_COUNT}')
for idx in range(GPU_COUNT):
    print(f'  [{idx}] {torch.cuda.get_device_name(idx)}')
print(f'AMP enabled: {USE_AMP}')
print(f'DataParallel enabled: {USE_MULTI_GPU}')

## Dataset Path Resolution

The local expected layout is:

```text
data/Zhao2024/{Normal,Stage1,Stage2,Stage3,laser scars}
data/Agrawal2021/HVDROPDB-BV/...
```

On Kaggle, the notebook searches under `/kaggle/input` for folders with the same internal structure.

In [ ]:
def has_zhao_structure(path: Path) -> bool:
    return path.exists() and all((path / cls).is_dir() for cls in cfg.class_names)


def has_agrawal_structure(path: Path) -> bool:
    return path.exists() and (path / 'HVDROPDB-BV').is_dir()


def shallow_dirs(root: Path, max_depth: int = 6) -> List[Path]:
    if not root.exists():
        return []
    results = [root]
    frontier = [(root, 0)]
    while frontier:
        current, depth = frontier.pop(0)
        if depth >= max_depth:
            continue
        try:
            children = [p for p in current.iterdir() if p.is_dir()]
        except PermissionError:
            continue
        results.extend(children)
        frontier.extend((child, depth + 1) for child in children)
    return results


def find_zhao_root() -> Path:
    candidates = [PROJECT_ROOT / 'data' / 'Zhao2024', INPUT_ROOT / 'Zhao2024']
    for root in USER_DATA_ROOTS:
        candidates.extend([root, root / 'Zhao2024', root / 'data' / 'Zhao2024'])
    candidates.extend(shallow_dirs(INPUT_ROOT, max_depth=6))
    for candidate in candidates:
        if has_zhao_structure(candidate):
            return candidate
    inspected = '\n'.join(str(p) for p in candidates[:80])
    raise FileNotFoundError('Could not find Zhao2024 folder with Normal, Stage1, Stage2, and Stage3 subfolders. First inspected candidates:\n' + inspected)


def find_agrawal_root() -> Optional[Path]:
    candidates = [PROJECT_ROOT / 'data' / 'Agrawal2021', INPUT_ROOT / 'Agrawal2021']
    for root in USER_DATA_ROOTS:
        candidates.extend([root, root / 'Agrawal2021', root / 'data' / 'Agrawal2021'])
    candidates.extend(shallow_dirs(INPUT_ROOT, max_depth=6))
    for candidate in candidates:
        if has_agrawal_structure(candidate):
            return candidate
    return None

ZHAO_ROOT = find_zhao_root()
AGRAWAL_ROOT = find_agrawal_root()

print(f'ZHAO_ROOT   : {ZHAO_ROOT}')
print(f'AGRAWAL_ROOT: {AGRAWAL_ROOT}')

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


def build_zhao_index(root: Path) -> pd.DataFrame:
    rows = []
    for class_name in cfg.class_names:
        class_dir = root / class_name
        files = sorted([p for p in class_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS])
        for path in files:
            rows.append({
                'path': str(path),
                'filename': path.name,
                'label': class_name,
                'label_id': CLASS_TO_ID[class_name],
            })
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f'No Zhao2024 images found under {root}')
    return df

zhao_df = build_zhao_index(ZHAO_ROOT)
display(zhao_df.head())
display(zhao_df['label'].value_counts().reindex(cfg.class_names))
print(f'Total images used: {len(zhao_df)}')
print(f'Excluded classes: {cfg.excluded_classes}')

In [ ]:
SPLIT_CSV = OUTPUT_SPLITS_DIR / 'zhao_image_level_split.csv'


def make_stratified_split(df: pd.DataFrame, split_csv: Path, seed: int) -> pd.DataFrame:
    if split_csv.exists():
        split_df = pd.read_csv(split_csv)
        print(f'Loaded existing split: {split_csv}')
        return split_df

    train_df, temp_df = train_test_split(
        df,
        test_size=(1.0 - cfg.train_ratio),
        stratify=df['label_id'],
        random_state=seed,
    )
    relative_test_ratio = cfg.test_ratio / (cfg.val_ratio + cfg.test_ratio)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test_ratio,
        stratify=temp_df['label_id'],
        random_state=seed,
    )

    train_df = train_df.copy(); train_df['split'] = 'train'
    val_df = val_df.copy(); val_df['split'] = 'val'
    test_df = test_df.copy(); test_df['split'] = 'test'
    split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    split_df = split_df.sort_values(['split', 'label', 'filename']).reset_index(drop=True)
    split_df.to_csv(split_csv, index=False)
    print(f'Saved split: {split_csv}')
    return split_df

split_df = make_stratified_split(zhao_df, SPLIT_CSV, cfg.seed)
display(pd.crosstab(split_df['split'], split_df['label']).reindex(columns=cfg.class_names))
print('Note: this is an image-level split. Zhao2024 patient/eye identifiers are not available in local filenames, so patient-level leakage cannot be fully ruled out.')

In [ ]:
def read_rgb(path: Path | str) -> np.ndarray:
    path = str(path)
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f'Could not read image: {path}')
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def write_rgb(path: Path | str, image: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    image = np.clip(image, 0, 255).astype(np.uint8)
    cv2.imwrite(str(path), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))


def resize_rgb(image: np.ndarray, size: int = cfg.image_size, interpolation: int = cv2.INTER_AREA) -> np.ndarray:
    return cv2.resize(image, (size, size), interpolation=interpolation)


def resize_max_side(image: np.ndarray, max_side: int, interpolation: int = cv2.INTER_AREA) -> np.ndarray:
    height, width = image.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    if scale >= 1.0:
        return image.copy()
    new_width = int(round(width * scale))
    new_height = int(round(height * scale))
    return cv2.resize(image, (new_width, new_height), interpolation=interpolation)


def to_uint8(image: np.ndarray) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    if arr.max() <= 1.0:
        arr = arr * 255.0
    return np.clip(arr, 0, 255).astype(np.uint8)


def estimate_fov_mask(rgb: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    threshold = max(5, int(np.percentile(gray, 2)))
    mask = gray > threshold
    mask = ndi.binary_fill_holes(mask)
    mask = binary_closing(mask, disk(5))
    labels, num = ndi.label(mask)
    if num > 0:
        sizes = ndi.sum(mask, labels, index=np.arange(1, num + 1))
        largest_label = int(np.argmax(sizes) + 1)
        mask = labels == largest_label
    return mask.astype(bool)

def erode_binary_mask(mask: np.ndarray, radius: int) -> np.ndarray:
    if radius <= 0:
        return mask.astype(bool)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    eroded = cv2.erode(mask.astype(np.uint8), kernel, iterations=1).astype(bool)
    return eroded


def remove_border_components(mask: np.ndarray, border_px: int = 2) -> np.ndarray:
    labels, num = ndi.label(mask.astype(bool))
    if num == 0:
        return mask.astype(bool)
    border = np.zeros(mask.shape, dtype=bool)
    border[:border_px, :] = True
    border[-border_px:, :] = True
    border[:, :border_px] = True
    border[:, -border_px:] = True
    touching = np.unique(labels[border])
    cleaned = mask.astype(bool).copy()
    for label in touching:
        if label != 0:
            cleaned[labels == label] = False
    return cleaned


def show_image_grid(images: List[np.ndarray], titles: List[str], cols: int = 4, figsize: Tuple[int, int] = (14, 8)) -> None:
    rows = int(math.ceil(len(images) / cols))
    plt.figure(figsize=figsize)
    for idx, (image, title) in enumerate(zip(images, titles), start=1):
        ax = plt.subplot(rows, cols, idx)
        if image.ndim == 2:
            ax.imshow(image, cmap='gray')
        else:
            ax.imshow(image)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Inlined Fixed Vessel Pipeline

This cell contains the fixed classical vessel pipeline and freezes the `normalization_residual_tuned` baseline inside the notebook.


In [ ]:
VesselnessMode = Literal["almeida", "almeida_paper", "ensemble", "legacy_tophat"]
ThresholdMethod = Literal["triangle", "otsu", "percentile", "hybrid"]
MorphologyMode = Literal["none", "open", "close", "open_close"]


@dataclass(frozen=True)
class VesselPipelineConfig:
    process_max_side: int = 768
    vesselness_mode: VesselnessMode = "almeida_paper"
    clahe_clip: float = 2.5
    green_clahe_clip: float = 2.2
    green_clahe_tile_grid: int = 16
    background_sigma: float = 30.0
    threshold_method: ThresholdMethod = "percentile"
    target_density: float = 0.10
    triangle_nbins: int = 4
    fov_erode_px: int = 12
    min_component_area: int = 50
    hole_size: int = 0
    morphology: MorphologyMode = "none"
    od_suppression: bool = False
    od_soft_penalty: float = 0.35
    bilateral_d: int = 9
    bilateral_sigma_color: float = 40.0
    bilateral_sigma_space: float = 9.0
    final_skeletonize: bool = False
    skeleton_min_area: int = 20
    skeleton_dilate_iter: int = 1
    auto_binary_selection: bool = False
    top_hat_weight: float = 0.05
    matched_weight: float = 0.25
    frangi_weight: float = 0.20
    jerman_weight: float = 0.50


NORMALIZATION_RESIDUAL_TUNED_BASELINE = VesselPipelineConfig()


@dataclass
class VesselFeatureResult:
    soft: np.ndarray
    fov: np.ndarray
    od_mask: np.ndarray
    enhanced_green: np.ndarray


@dataclass
class VesselDebugResult:
    raw_rgb: np.ndarray
    fov: np.ndarray
    cielab_rgb: np.ndarray
    green: np.ndarray
    green_filtered: np.ndarray
    background_fixed: np.ndarray
    normalized_fixed: np.ndarray
    background_new: np.ndarray
    normalized_new: np.ndarray
    normalized_clamped: np.ndarray
    green_light_filtered: np.ndarray
    normalized_light_residual: np.ndarray
    normalized_weak_residual: np.ndarray
    normalized_light_weak_residual: np.ndarray
    normalized_clahe3_weak_residual: np.ndarray
    bsgmf: np.ndarray
    top_hat: np.ndarray
    frangi: np.ndarray
    hessian: np.ndarray
    combined: np.ndarray
    triangle: np.ndarray
    triangle_clean: np.ndarray
    final_binary: np.ndarray
    od_mask: np.ndarray


def resize_max_side(image: np.ndarray, max_side: int) -> np.ndarray:
    height, width = image.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    if scale >= 1.0:
        return image.copy()
    new_width = int(round(width * scale))
    new_height = int(round(height * scale))
    return cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)


def estimate_fov_mask(rgb: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    threshold = max(3, int(np.percentile(gray, 1)))
    mask = (gray > threshold).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = ndi.binary_fill_holes(mask > 0).astype(np.uint8)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n_labels > 1:
        largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        mask = (labels == largest).astype(np.uint8)
    return mask.astype(bool)


def erode_mask(mask: np.ndarray, radius: int) -> np.ndarray:
    if radius <= 0:
        return mask.astype(bool)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    return cv2.erode(mask.astype(np.uint8), kernel, iterations=1).astype(bool)


def normalize01(image: np.ndarray, mask: np.ndarray | None = None) -> np.ndarray:
    values = image[mask] if mask is not None else image.reshape(-1)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.zeros(image.shape, dtype=np.float32)
    low, high = np.percentile(values, [1, 99])
    if high <= low:
        low, high = float(values.min()), float(values.max())
    if high <= low:
        return np.zeros(image.shape, dtype=np.float32)
    output = (image.astype(np.float32) - float(low)) / float(high - low)
    return np.clip(output, 0.0, 1.0).astype(np.float32)


def rank_normalize(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    output = np.zeros(image.shape, dtype=np.float32)
    values = image[mask].astype(np.float32)
    valid = np.isfinite(values)
    if values.size == 0 or valid.sum() < 2:
        return output
    ranks = np.empty(values.shape, dtype=np.float32)
    order = np.argsort(values[valid], kind="mergesort")
    ranked = np.empty(order.shape, dtype=np.float32)
    ranked[order] = np.linspace(0.0, 1.0, num=order.size, dtype=np.float32)
    ranks[valid] = ranked
    ranks[~valid] = 0.0
    output[mask] = ranks
    return output


def contrast_stretch_uint8(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    stretched = normalize01(image.astype(np.float32), mask)
    return np.clip(stretched * 255.0, 0, 255).astype(np.uint8)


def enhance_cielab_rgb(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> tuple[np.ndarray, np.ndarray]:
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=float(config.clahe_clip), tileGridSize=(8, 8))
    l_clahe = clahe.apply(l_channel)

    # Keep this aggressive; the background-normalization step below controls
    # the illumination field before vessel filters see the image.
    l_blur = cv2.GaussianBlur(l_clahe, (0, 0), 1.2)
    l_sharp = cv2.addWeighted(l_clahe, 1.35, l_blur, -0.35, 0)
    l_sharp = np.clip(l_sharp, 0, 255).astype(np.uint8)

    enhanced_rgb = cv2.cvtColor(cv2.merge([l_sharp, a_channel, b_channel]), cv2.COLOR_LAB2RGB)
    enhanced_rgb[~fov] = 0
    return enhanced_rgb, l_sharp


def estimate_background_mean(green: np.ndarray, fov: np.ndarray, kernel_size: int) -> np.ndarray:
    kernel_size = max(3, int(round(kernel_size)))
    filled = green.astype(np.float32).copy()
    if np.any(fov):
        filled[~fov] = float(np.median(filled[fov]))
    background = cv2.blur(filled, (kernel_size, kernel_size), borderType=cv2.BORDER_REFLECT)
    background[~fov] = 0
    return np.clip(background, 0, 255).astype(np.uint8)


def normalize_green_with_background(green: np.ndarray, background: np.ndarray, fov: np.ndarray) -> np.ndarray:
    output = np.zeros(green.shape, dtype=np.float32)
    if not np.any(fov):
        return output.astype(np.uint8)

    green_float = green.astype(np.float32)
    background_float = np.maximum(background.astype(np.float32), 1.0)
    scale = float(np.median(background_float[fov]))
    corrected = green_float * scale / background_float
    values = corrected[fov]
    low, high = np.percentile(values, [0.5, 99.5])
    if high <= low:
        low, high = float(values.min()), float(values.max())
    if high <= low:
        return output.astype(np.uint8)
    output[fov] = (corrected[fov] - low) / float(high - low)
    output = np.clip(output, 0.0, 1.0)
    output = 32.0 + 223.0 * output
    output[~fov] = 0
    return np.clip(output, 0, 255).astype(np.uint8)


def estimate_background_multiscale(green: np.ndarray, fov: np.ndarray, base_kernel_size: int) -> np.ndarray:
    base_kernel_size = max(3, int(round(base_kernel_size)))
    if base_kernel_size % 2 == 0:
        base_kernel_size += 1
    large_kernel_size = max(base_kernel_size * 2 + 1, int(round(min(green.shape) * 0.12)))
    if large_kernel_size % 2 == 0:
        large_kernel_size += 1

    filled = green.astype(np.float32).copy()
    if np.any(fov):
        filled[~fov] = float(np.median(filled[fov]))

    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (base_kernel_size, base_kernel_size))
    vessel_suppressed = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, close_kernel)
    background_base = cv2.blur(vessel_suppressed, (base_kernel_size, base_kernel_size), borderType=cv2.BORDER_REFLECT)
    background_large = cv2.blur(vessel_suppressed, (large_kernel_size, large_kernel_size), borderType=cv2.BORDER_REFLECT)
    background = 0.65 * background_base + 0.35 * background_large
    background[~fov] = 0
    return np.clip(background, 0, 255).astype(np.uint8)


def normalize_green_with_background_clamped(
    green: np.ndarray,
    background: np.ndarray,
    fov: np.ndarray,
    stats_erode_px: int = 12,
    gain_min: float = 0.85,
    gain_max: float = 1.20,
) -> np.ndarray:
    output = np.zeros(green.shape, dtype=np.float32)
    stats_fov = erode_mask(fov, int(stats_erode_px))
    if not np.any(stats_fov):
        stats_fov = fov.astype(bool)
    if not np.any(stats_fov):
        return output.astype(np.uint8)

    green_float = green.astype(np.float32)
    background_float = np.maximum(background.astype(np.float32), 1.0)
    scale = float(np.median(background_float[stats_fov]))
    gain = np.clip(scale / background_float, float(gain_min), float(gain_max))
    corrected = green_float * gain

    values = corrected[stats_fov]
    low, high = np.percentile(values, [0.5, 99.5])
    if high <= low:
        low, high = float(values.min()), float(values.max())
    if high <= low:
        return output.astype(np.uint8)
    output[fov] = (corrected[fov] - low) / float(high - low)
    output = np.clip(output, 0.0, 1.0)
    output = 24.0 + 200.0 * output
    output[~fov] = 0
    return np.clip(output, 0, 255).astype(np.uint8)


def normalize_green_with_background_mapped(
    green: np.ndarray,
    background: np.ndarray,
    fov: np.ndarray,
    stats_erode_px: int = 12,
    gamma: float = 1.3,
) -> np.ndarray:
    output = np.zeros(green.shape, dtype=np.float32)
    stats_fov = erode_mask(fov, int(stats_erode_px))
    if not np.any(stats_fov):
        stats_fov = fov.astype(bool)
    if not np.any(stats_fov):
        return output.astype(np.uint8)

    green_float = green.astype(np.float32)
    background_float = np.maximum(background.astype(np.float32), 1.0)
    scale = float(np.median(background_float[stats_fov]))
    corrected = green_float * scale / background_float

    values = corrected[stats_fov]
    low, high = np.percentile(values, [2.0, 98.0])
    if high <= low:
        low, high = float(values.min()), float(values.max())
    if high <= low:
        return output.astype(np.uint8)

    mapped = np.clip((corrected - low) / float(high - low), 0.0, 1.0)
    mapped = np.power(mapped, float(gamma))
    output[fov] = 40.0 + 185.0 * mapped[fov]
    output[~fov] = 0
    return np.clip(output, 0, 255).astype(np.uint8)


def normalize_green_with_background_residual(
    green: np.ndarray,
    background: np.ndarray,
    fov: np.ndarray,
    stats_erode_px: int = 12,
    vessel_contrast: float = 118.0,
    background_contrast: float = 34.0,
    vessel_gamma: float = 0.72,
    background_gamma: float = 1.55,
) -> np.ndarray:
    output = np.zeros(green.shape, dtype=np.float32)
    stats_fov = erode_mask(fov, int(stats_erode_px))
    if not np.any(stats_fov):
        stats_fov = fov.astype(bool)
    if not np.any(stats_fov):
        return output.astype(np.uint8)

    residual = green.astype(np.float32) - background.astype(np.float32)
    values = residual[stats_fov]
    center = float(np.median(values))
    low, high = np.percentile(values, [1.0, 99.0])
    negative_scale = max(center - float(low), 1.0)
    positive_scale = max(float(high) - center, 1.0)
    if negative_scale <= 0 and positive_scale <= 0:
        return output.astype(np.uint8)

    dark_response = np.clip((center - residual) / negative_scale, 0.0, 1.0)
    bright_response = np.clip((residual - center) / positive_scale, 0.0, 1.0)

    # Dark vessels are clinically useful, so weak dark residuals get a boost.
    # Bright residuals are mostly illumination/texture, so keep them compressed.
    dark_response = np.power(dark_response, float(vessel_gamma))
    bright_response = np.power(bright_response, float(background_gamma))
    output[fov] = 142.0 - float(vessel_contrast) * dark_response[fov] + float(background_contrast) * bright_response[fov]
    output[~fov] = 0
    return np.clip(output, 0, 255).astype(np.uint8)


def bilateral_filter_green(green: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> np.ndarray:
    d = int(config.bilateral_d)
    if d <= 1:
        return green.copy()
    if d % 2 == 0:
        d += 1

    filled = green.copy()
    if np.any(fov):
        filled[~fov] = int(np.median(filled[fov]))
    filtered = cv2.bilateralFilter(
        filled,
        d=d,
        sigmaColor=float(config.bilateral_sigma_color),
        sigmaSpace=float(config.bilateral_sigma_space),
    )
    filtered[~fov] = 0
    return filtered.astype(np.uint8)


def preprocess_green_channel(
    rgb: np.ndarray,
    fov: np.ndarray,
    config: VesselPipelineConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    cielab_rgb, _ = enhance_cielab_rgb(rgb, fov, config)
    green_tile = max(2, int(config.green_clahe_tile_grid))
    clahe = cv2.createCLAHE(clipLimit=float(config.green_clahe_clip), tileGridSize=(green_tile, green_tile))
    green = cielab_rgb[:, :, 1]
    green_clahe = clahe.apply(green)
    green_clahe[~fov] = 0
    green_filtered = bilateral_filter_green(green_clahe, fov, config)
    background_fixed = estimate_background_mean(green_filtered, fov, int(round(config.background_sigma)))
    normalized_fixed = normalize_green_with_background(green_filtered, background_fixed, fov)
    background_new = background_fixed
    normalized_new = normalize_green_with_background_residual(
        green_filtered,
        background_new,
        fov,
        stats_erode_px=int(config.fov_erode_px),
    )
    return cielab_rgb, green_clahe, green_filtered, background_fixed, normalized_fixed, background_new, normalized_new, normalized_new


def enhance_green_channel(rgb: np.ndarray, config: VesselPipelineConfig) -> tuple[np.ndarray, np.ndarray]:
    working_rgb = resize_max_side(rgb, config.process_max_side)
    fov = estimate_fov_mask(working_rgb)

    *_, enhanced_green = preprocess_green_channel(working_rgb, fov, config)
    return enhanced_green, fov


def modified_tophat(inverted: np.ndarray, fov: np.ndarray) -> np.ndarray:
    maps = []
    for kernel_size in (5, 7, 9, 13, 17, 23, 31):
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        opened = cv2.morphologyEx(inverted, cv2.MORPH_OPEN, kernel)
        top_hat = cv2.subtract(inverted, opened).astype(np.float32)
        maps.append(top_hat)
    response = np.max(np.stack(maps, axis=0), axis=0)
    response = cv2.GaussianBlur(response, (0, 0), 0.8)
    return normalize01(response, fov)


def line_kernel(size: int, angle_deg: int, sigma: float) -> np.ndarray:
    center = size // 2
    yy, xx = np.mgrid[:size, :size].astype(np.float32) - center
    radians = np.deg2rad(angle_deg)
    x_rot = xx * np.cos(radians) + yy * np.sin(radians)
    y_rot = -xx * np.sin(radians) + yy * np.cos(radians)
    kernel = np.exp(-(y_rot**2) / (2.0 * sigma**2))
    kernel[np.abs(x_rot) > center] = 0
    kernel -= kernel.mean()
    norm = np.sum(np.abs(kernel))
    if norm > 0:
        kernel /= norm
    return kernel.astype(np.float32)


def matched_filter_response(inverted_float: np.ndarray, fov: np.ndarray) -> np.ndarray:
    response = np.zeros(inverted_float.shape, dtype=np.float32)
    for size, sigma in ((9, 1.2), (15, 1.8), (21, 2.4), (31, 3.2)):
        for angle in range(0, 180, 10):
            kernel = line_kernel(size, angle, sigma)
            filtered = cv2.filter2D(inverted_float, cv2.CV_32F, kernel, borderType=cv2.BORDER_REFLECT)
            response = np.maximum(response, filtered)
    response[~fov] = 0
    return normalize01(response, fov)


def preprocess_green_channel_almeida_paper(
    rgb: np.ndarray,
    fov: np.ndarray,
    config: VesselPipelineConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    cielab_rgb, _ = enhance_cielab_rgb(rgb, fov, config)
    green = cielab_rgb[:, :, 1].copy()
    green[~fov] = 0
    background = estimate_background_mean(green, fov, int(round(config.background_sigma)))
    normalized = normalize_green_with_background(green, background, fov)
    return cielab_rgb, green, background, normalized


def matched_filter_response_almeida_paper(inverted_float: np.ndarray, fov: np.ndarray) -> np.ndarray:
    response = np.zeros(inverted_float.shape, dtype=np.float32)
    for angle in range(0, 180, 15):
        kernel = line_kernel(15, angle, 2.0)
        filtered = cv2.filter2D(inverted_float, cv2.CV_32F, kernel, borderType=cv2.BORDER_REFLECT)
        response = np.maximum(response, filtered)
    response[~fov] = 0
    return normalize01(response, fov)


def modified_tophat_almeida_paper(bright_vessels: np.ndarray, fov: np.ndarray) -> np.ndarray:
    source = np.clip(normalize01(bright_vessels.astype(np.float32), fov) * 255.0, 0, 255).astype(np.uint8)
    close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    maps = []
    for radius in (1, 2, 3):
        open_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
        closed = cv2.morphologyEx(source, cv2.MORPH_CLOSE, close_kernel)
        opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, open_kernel)
        limited = np.minimum(opened, source)
        maps.append(cv2.subtract(source, limited).astype(np.float32))
    response = np.mean(np.stack(maps, axis=0), axis=0)
    response[~fov] = 0
    return normalize01(response, fov)


def jerman_vesselness_almeida_paper(bright_vessels: np.ndarray, fov: np.ndarray, sigmas: tuple[float, ...] = (1.0, 3.0, 5.0, 7.0), tau: float = 0.90) -> np.ndarray:
    image = bright_vessels.astype(np.float32)
    output = np.zeros(image.shape, dtype=np.float32)
    for sigma in sigmas:
        hessian = hessian_matrix(image, sigma=sigma, order="rc", use_gaussian_derivatives=True)
        eig_small, eig_large = hessian_matrix_eigvals(hessian)
        vessel_strength = -eig_large.astype(np.float32)
        positive = (vessel_strength > 0) & fov
        if not np.any(positive):
            continue
        lambda_p = float(tau) * float(np.percentile(vessel_strength[positive], 99.5))
        if lambda_p <= 0:
            continue
        rho = np.clip(vessel_strength / lambda_p, 0.0, 1.0)
        blob_penalty = np.exp(-(np.abs(eig_small) ** 2) / (2.0 * (vessel_strength ** 2 + 1e-6)))
        vessel = (rho ** 2) * (3.0 - 2.0 * rho) * blob_penalty
        vessel[~positive] = 0
        output = np.maximum(output, vessel.astype(np.float32))
    output[~fov] = 0
    return normalize01(output, fov)


def almeida_paper_vessel_maps(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> dict[str, np.ndarray]:
    cielab_rgb, green, background, normalized = preprocess_green_channel_almeida_paper(rgb, fov, config)
    inverted = 255 - normalized
    inverted_float = normalize01(inverted.astype(np.float32), fov)
    bsgmf = matched_filter_response_almeida_paper(inverted_float, fov)
    top_hat = modified_tophat_almeida_paper(bsgmf, fov)
    frangi_map = normalize01(frangi(top_hat.astype(np.float32), sigmas=(1, 3, 5, 7), alpha=0.5, beta=15.0, black_ridges=False), fov)
    jerman_map = jerman_vesselness_almeida_paper(top_hat, fov)
    combined = normalize01(0.70 * frangi_map + 0.30 * jerman_map, fov)
    return {
        'cielab_rgb': cielab_rgb,
        'green': green,
        'background': background,
        'normalized': normalized,
        'bsgmf': bsgmf,
        'top_hat': top_hat,
        'frangi': frangi_map,
        'jerman': jerman_map,
        'combined': combined,
    }


def jerman_vesselness(bright_vessels: np.ndarray, fov: np.ndarray, sigmas: tuple[float, ...] = (1.0, 1.6, 2.4, 3.2, 4.8)) -> np.ndarray:
    """Approximate Jerman-style 2D vesselness for bright tubular structures.

    The implementation follows the same practical goal as Jerman vesselness:
    reduce non-uniform response across vessel calibers by normalizing Hessian
    eigenvalues per scale before combining them.
    """

    image = bright_vessels.astype(np.float32)
    output = np.zeros(image.shape, dtype=np.float32)
    for sigma in sigmas:
        hessian = hessian_matrix(image, sigma=sigma, order="rc", use_gaussian_derivatives=True)
        eig_small, eig_large = hessian_matrix_eigvals(hessian)
        lambda2 = -eig_large
        lambda1 = np.abs(eig_small)
        positive = lambda2 > 0
        if not np.any(positive & fov):
            continue
        lambda2_norm = np.zeros_like(lambda2, dtype=np.float32)
        scale_max = np.percentile(lambda2[positive & fov], 99)
        if scale_max <= 0:
            continue
        lambda2_norm[positive] = np.clip(lambda2[positive] / scale_max, 0.0, 1.0)
        blob_penalty = np.exp(-(lambda1**2) / (2.0 * (lambda2**2 + 1e-6)))
        vessel = lambda2_norm * blob_penalty
        vessel[~positive] = 0
        output = np.maximum(output, vessel.astype(np.float32))
    output[~fov] = 0
    return normalize01(output, fov)


def safe_filter_call(name: str, image: np.ndarray) -> np.ndarray:
    if name == "frangi":
        return frangi(image, sigmas=(1, 2, 3, 4, 5), black_ridges=False)
    if name == "sato":
        return sato(image, sigmas=(1, 2, 3, 4, 5), black_ridges=False)
    if name == "meijering":
        return meijering(image, sigmas=(1, 2, 3, 4, 5), black_ridges=False)
    raise ValueError(f"Unknown vessel filter: {name}")


def fuse_vessel_responses(
    top_hat: np.ndarray,
    matched: np.ndarray,
    frangi_map: np.ndarray | None,
    jerman_map: np.ndarray,
    fov: np.ndarray,
    sato_map: np.ndarray | None = None,
    meijering_map: np.ndarray | None = None,
    local_dark: np.ndarray | None = None,
    mode: VesselnessMode = "almeida",
    config: VesselPipelineConfig = VesselPipelineConfig(),
) -> np.ndarray:
    """Fuse normalized response maps without flattening their histograms.

    Rank normalization was useful for making weak filters contribute, but on
    these ROP images it also promotes diffuse texture. The thresholded mask evaluation
    benefits more from preserving absolute response strength.
    """

    if mode == "legacy_tophat":
        return normalize01(top_hat, fov)
    if mode == "almeida":
        if frangi_map is None:
            raise ValueError("frangi_map is required for Almeida fusion")
        total_weight = max(
            float(config.top_hat_weight)
            + float(config.matched_weight)
            + float(config.frangi_weight)
            + float(config.jerman_weight),
            1e-6,
        )
        fused = (
            float(config.top_hat_weight) * top_hat
            + float(config.matched_weight) * matched
            + float(config.frangi_weight) * frangi_map
            + float(config.jerman_weight) * jerman_map
        ) / total_weight
    elif mode == "ensemble":
        if frangi_map is None or sato_map is None or meijering_map is None or local_dark is None:
            raise ValueError("all filter maps are required for ensemble fusion")
        fused = (
            0.14 * top_hat
            + 0.34 * matched
            + 0.12 * frangi_map
            + 0.22 * jerman_map
            + 0.08 * sato_map
            + 0.06 * meijering_map
            + 0.04 * local_dark
        )
    else:
        raise ValueError(f"Unknown vesselness mode: {mode}")

    fused = cv2.GaussianBlur(fused.astype(np.float32), (0, 0), 0.6)
    fused[~fov] = 0
    return normalize01(fused, fov)


def estimate_optic_disc_mask(rgb: np.ndarray, fov: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l_channel = lab[:, :, 0].astype(np.float32)
    green = rgb[:, :, 1].astype(np.float32)
    score = normalize01(0.65 * l_channel + 0.35 * green, fov)
    if np.count_nonzero(fov) == 0:
        return np.zeros(fov.shape, dtype=bool)

    threshold = np.percentile(score[fov], 97)
    candidate = (score >= threshold) & fov
    candidate = ndi.binary_fill_holes(candidate)
    candidate = cv2.morphologyEx(candidate.astype(np.uint8), cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))).astype(bool)
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(candidate.astype(np.uint8), 8)
    if n_labels <= 1:
        return np.zeros(fov.shape, dtype=bool)

    height, width = fov.shape
    min_area = 0.0025 * height * width
    max_area = 0.08 * height * width
    best_label = 0
    best_score = -np.inf
    for label in range(1, n_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        if not (min_area <= area <= max_area):
            continue
        x, y, w, h, _ = stats[label]
        aspect = min(w, h) / max(w, h)
        compactness = aspect * area / max(w * h, 1)
        cx, cy = centroids[label]
        center_bias = 1.0 - min(1.0, abs(cx - width * 0.5) / (width * 0.55))
        label_score = compactness + 0.2 * center_bias + float(score[labels == label].mean())
        if label_score > best_score:
            best_score = label_score
            best_label = label
    if best_label == 0:
        return np.zeros(fov.shape, dtype=bool)

    disc = labels == best_label
    radius = max(7, int(round(np.sqrt(stats[best_label, cv2.CC_STAT_AREA] / np.pi) * 1.35)))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    disc = cv2.dilate(disc.astype(np.uint8), kernel, iterations=1).astype(bool)
    return disc & fov


def compute_vessel_features(rgb: np.ndarray, config: VesselPipelineConfig = VesselPipelineConfig()) -> VesselFeatureResult:
    working_rgb = resize_max_side(rgb, config.process_max_side)
    fov = estimate_fov_mask(working_rgb)
    if config.vesselness_mode == "almeida_paper":
        maps = almeida_paper_vessel_maps(working_rgb, fov, config)
        soft = maps['combined']
        od_mask = estimate_optic_disc_mask(working_rgb, fov) if config.od_suppression else np.zeros(fov.shape, dtype=bool)
        if config.od_suppression and od_mask.any():
            soft = soft.copy()
            soft[od_mask] *= float(config.od_soft_penalty)
            soft = normalize01(soft, fov)
        soft[~fov] = 0
        return VesselFeatureResult(soft=soft.astype(np.float32), fov=fov.astype(bool), od_mask=od_mask.astype(bool), enhanced_green=maps['normalized'])

    *_, enhanced_green = preprocess_green_channel(working_rgb, fov, config)
    inverted = 255 - enhanced_green
    inverted_float = normalize01(inverted.astype(np.float32), fov)

    top_hat = modified_tophat(inverted, fov)
    matched = matched_filter_response(inverted_float, fov)

    if config.vesselness_mode == "legacy_tophat":
        soft = top_hat
    else:
        frangi_map = normalize01(safe_filter_call("frangi", inverted_float), fov)
        jerman_map = jerman_vesselness(inverted_float, fov)
        if config.vesselness_mode == "almeida":
            soft = fuse_vessel_responses(top_hat, matched, frangi_map, jerman_map, fov, mode=config.vesselness_mode, config=config)
        elif config.vesselness_mode == "ensemble":
            sato_map = normalize01(safe_filter_call("sato", inverted_float), fov)
            meijering_map = normalize01(safe_filter_call("meijering", inverted_float), fov)
            local_dark = normalize01(inverted_float - cv2.GaussianBlur(inverted_float, (0, 0), 7.0), fov)
            soft = fuse_vessel_responses(
                top_hat,
                matched,
                frangi_map,
                jerman_map,
                fov,
                sato_map=sato_map,
                meijering_map=meijering_map,
                local_dark=local_dark,
                mode=config.vesselness_mode,
                config=config,
            )
        else:
            raise ValueError(f"Unknown vesselness mode: {config.vesselness_mode}")

    od_mask = estimate_optic_disc_mask(working_rgb, fov) if config.od_suppression else np.zeros(fov.shape, dtype=bool)
    if config.od_suppression and od_mask.any():
        soft = soft.copy()
        soft[od_mask] *= float(config.od_soft_penalty)
        soft = normalize01(soft, fov)
    soft[~fov] = 0
    return VesselFeatureResult(soft=soft.astype(np.float32), fov=fov.astype(bool), od_mask=od_mask.astype(bool), enhanced_green=enhanced_green)


def threshold_from_values(values: np.ndarray, method: ThresholdMethod, target_density: float, triangle_nbins: int = 256) -> float:
    values = values[np.isfinite(values)]
    values = values[values > 0]
    if values.size < 16 or float(values.max() - values.min()) < 1e-6:
        return 0.5
    percentile_threshold = float(np.percentile(values, 100.0 * (1.0 - target_density)))
    if method == "percentile":
        return percentile_threshold
    try:
        if method == "triangle":
            return float(threshold_triangle(values, nbins=max(2, int(triangle_nbins))))
        if method == "otsu":
            return float(threshold_otsu(values))
        if method == "hybrid":
            triangle = float(threshold_triangle(values, nbins=max(2, int(triangle_nbins))))
            return min(triangle, percentile_threshold)
    except ValueError:
        return percentile_threshold
    raise ValueError(f"Unknown threshold method: {method}")


def remove_border_components(mask: np.ndarray, border_px: int = 3) -> np.ndarray:
    n_labels, labels, _, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    if n_labels <= 1:
        return mask.astype(bool)
    border = np.zeros(mask.shape, dtype=bool)
    border[:border_px, :] = True
    border[-border_px:, :] = True
    border[:, :border_px] = True
    border[:, -border_px:] = True
    output = mask.astype(bool).copy()
    for label in np.unique(labels[border]):
        if label != 0:
            output[labels == label] = False
    return output


def keep_components_at_least(mask: np.ndarray, min_area: int) -> np.ndarray:
    if min_area <= 1:
        return mask.astype(bool)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    output = np.zeros(mask.shape, dtype=bool)
    for label in range(1, n_labels):
        if stats[label, cv2.CC_STAT_AREA] >= min_area:
            output[labels == label] = True
    return output


def remove_optic_disc_components(mask: np.ndarray, od_mask: np.ndarray) -> np.ndarray:
    if not od_mask.any() or not mask.any():
        return mask.astype(bool)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    output = mask.astype(bool).copy()
    for label in range(1, n_labels):
        component = labels == label
        area = stats[label, cv2.CC_STAT_AREA]
        overlap = np.count_nonzero(component & od_mask) / max(area, 1)
        if overlap > 0.55 and area > 12:
            output[component] = False
    return output


def fill_holes_up_to(mask: np.ndarray, max_size: int) -> np.ndarray:
    if max_size <= 0:
        return mask.astype(bool)
    try:
        return remove_small_holes(mask, max_size=int(max_size))
    except TypeError:
        return remove_small_holes(mask, area_threshold=int(max_size))


def threshold_response_map(
    soft: np.ndarray,
    fov: np.ndarray,
    method: ThresholdMethod = "triangle",
    target_density: float = 0.14,
    fov_erode_px: int = 8,
    triangle_nbins: int = 256,
) -> np.ndarray:
    """Keep threshold-selected vessel evidence as a soft map.

    This is intentionally lighter than full binary postprocessing: thresholding
    chooses the support, but the returned image preserves the original
    vesselness magnitude for visual review and soft-score evaluation.
    """

    inner_fov = erode_mask(fov, int(fov_erode_px))
    working = soft.astype(np.float32).copy()
    working[~inner_fov] = 0.0
    threshold = threshold_from_values(working[inner_fov], method, float(target_density), triangle_nbins=triangle_nbins)
    response = np.zeros(working.shape, dtype=np.float32)
    response[(working >= threshold) & inner_fov] = working[(working >= threshold) & inner_fov]
    return normalize01(response, response > 0)


def threshold_debug_variants(soft: np.ndarray, fov: np.ndarray, fov_erode_px: int) -> list[tuple[str, np.ndarray]]:
    """Expose threshold choices before cleanup so vessel loss is inspectable."""

    variants: list[tuple[str, ThresholdMethod, float]] = [
        ("Triangle", "triangle", 0.14),
        ("P08", "percentile", 0.08),
        ("P10", "percentile", 0.10),
        ("P12", "percentile", 0.12),
        ("Hybrid10", "hybrid", 0.10),
    ]
    return [
        (
            label,
            threshold_response_map(
                soft,
                fov,
                method=method,
                target_density=target_density,
                fov_erode_px=fov_erode_px,
            ),
        )
        for label, method, target_density in variants
    ]


def clean_triangle_response(
    triangle: np.ndarray,
    fov: np.ndarray,
    min_component_area: int = 16,
    fov_erode_px: int = 8,
) -> np.ndarray:
    """Remove obvious speckle/rim artifacts while preserving vessel width."""

    inner_fov = erode_mask(fov, int(fov_erode_px))
    response = triangle.astype(np.float32).copy()
    response[~inner_fov] = 0.0
    support = response > 0
    support = keep_components_at_least(support, int(min_component_area))
    support = remove_edge_line_artifacts(support)
    response[~support] = 0.0
    return normalize01(response, response > 0)


def remove_edge_line_artifacts(mask: np.ndarray, edge_margin: int = 24, min_aspect: float = 5.0) -> np.ndarray:
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    if n_labels <= 1:
        return mask.astype(bool)

    height, width = mask.shape
    output = mask.astype(bool).copy()
    for label in range(1, n_labels):
        x, y, w, h, area = stats[label]
        near_edge = x <= edge_margin or y <= edge_margin or x + w >= width - edge_margin or y + h >= height - edge_margin
        if not near_edge:
            continue
        short_side = max(1, min(w, h))
        aspect = max(w, h) / short_side
        fill = area / max(w * h, 1)
        if aspect >= min_aspect and fill >= 0.12:
            output[labels == label] = False
    return output


def compute_vessel_debug_maps(rgb: np.ndarray, config: VesselPipelineConfig = VesselPipelineConfig()) -> VesselDebugResult:
    working_rgb = resize_max_side(rgb, config.process_max_side)
    fov = estimate_fov_mask(working_rgb)

    (
        cielab_rgb,
        green_clahe,
        green_filtered,
        background_fixed,
        normalized_fixed,
        background_new,
        normalized_new,
        enhanced_green,
    ) = preprocess_green_channel(working_rgb, fov, config)

    normalized_clamped = normalize_green_with_background_clamped(
        green_filtered,
        background_new,
        fov,
        stats_erode_px=int(config.fov_erode_px),
    )
    light_bilateral_config = replace(
        config,
        bilateral_d=5,
        bilateral_sigma_color=28.0,
        bilateral_sigma_space=5.0,
    )
    green_light_filtered = bilateral_filter_green(green_clahe, fov, light_bilateral_config)
    background_light = estimate_background_mean(green_light_filtered, fov, int(round(config.background_sigma)))
    normalized_light_residual = normalize_green_with_background_residual(
        green_light_filtered,
        background_light,
        fov,
        stats_erode_px=int(config.fov_erode_px),
    )
    weak_residual_kwargs = dict(
        stats_erode_px=int(config.fov_erode_px),
        vessel_contrast=130.0,
        background_contrast=25.0,
        vessel_gamma=0.60,
        background_gamma=1.65,
    )
    normalized_weak_residual = normalize_green_with_background_residual(
        green_filtered,
        background_new,
        fov,
        **weak_residual_kwargs,
    )
    normalized_light_weak_residual = normalize_green_with_background_residual(
        green_light_filtered,
        background_light,
        fov,
        **weak_residual_kwargs,
    )
    clahe3_config = replace(config, clahe_clip=3.0)
    (
        _,
        green_clahe3,
        green_filtered3,
        _,
        _,
        background3,
        _,
        _,
    ) = preprocess_green_channel(working_rgb, fov, clahe3_config)
    normalized_clahe3_weak_residual = normalize_green_with_background_residual(
        green_filtered3,
        background3,
        fov,
        **weak_residual_kwargs,
    )

    inverted = 255 - enhanced_green
    inverted_float = normalize01(inverted.astype(np.float32), fov)
    top_hat = modified_tophat(inverted, fov)
    bsgmf = matched_filter_response(inverted_float, fov)
    hessian = jerman_vesselness(inverted_float, fov)
    frangi_map = np.zeros_like(hessian, dtype=np.float32)

    if config.vesselness_mode == "almeida_paper":
        paper_maps = almeida_paper_vessel_maps(working_rgb, fov, config)
        cielab_rgb = paper_maps['cielab_rgb']
        green_clahe = paper_maps['green']
        green_filtered = paper_maps['green']
        background_fixed = paper_maps['background']
        normalized_fixed = paper_maps['normalized']
        background_new = paper_maps['background']
        normalized_new = paper_maps['normalized']
        enhanced_green = paper_maps['normalized']
        bsgmf = paper_maps['bsgmf']
        top_hat = paper_maps['top_hat']
        frangi_map = paper_maps['frangi']
        hessian = paper_maps['jerman']
        combined = paper_maps['combined']
    elif config.vesselness_mode == "legacy_tophat":
        combined = top_hat
    elif config.vesselness_mode == "almeida":
        frangi_map = normalize01(safe_filter_call("frangi", inverted_float), fov)
        combined = fuse_vessel_responses(top_hat, bsgmf, frangi_map, hessian, fov, mode=config.vesselness_mode, config=config)
    elif config.vesselness_mode == "ensemble":
        frangi_map = normalize01(safe_filter_call("frangi", inverted_float), fov)
        sato_map = normalize01(safe_filter_call("sato", inverted_float), fov)
        meijering_map = normalize01(safe_filter_call("meijering", inverted_float), fov)
        local_dark = normalize01(inverted_float - cv2.GaussianBlur(inverted_float, (0, 0), 7.0), fov)
        combined = fuse_vessel_responses(
            top_hat,
            bsgmf,
            frangi_map,
            hessian,
            fov,
            sato_map=sato_map,
            meijering_map=meijering_map,
            local_dark=local_dark,
            mode=config.vesselness_mode,
            config=config,
        )
    else:
        raise ValueError(f"Unknown vesselness mode: {config.vesselness_mode}")

    od_mask = estimate_optic_disc_mask(working_rgb, fov) if config.od_suppression else np.zeros(fov.shape, dtype=bool)
    if config.od_suppression and od_mask.any():
        combined = combined.copy()
        combined[od_mask] *= float(config.od_soft_penalty)
        combined = normalize01(combined, fov)

    triangle = threshold_response_map(
        combined,
        fov,
        method=config.threshold_method,
        target_density=float(config.target_density),
        fov_erode_px=max(0, int(config.fov_erode_px)),
        triangle_nbins=int(config.triangle_nbins),
    )
    triangle_clean = clean_triangle_response(
        triangle,
        fov,
        min_component_area=max(1, int(config.min_component_area)),
        fov_erode_px=max(0, int(config.fov_erode_px)),
    )
    final_binary = postprocess_vessel_map(combined, fov, config, od_mask)
    return VesselDebugResult(
        raw_rgb=working_rgb,
        fov=fov.astype(bool),
        cielab_rgb=cielab_rgb,
        green=green_clahe,
        green_filtered=green_filtered,
        background_fixed=background_fixed,
        normalized_fixed=normalized_fixed,
        background_new=background_new,
        normalized_new=normalized_new,
        normalized_clamped=normalized_clamped,
        green_light_filtered=green_light_filtered,
        normalized_light_residual=normalized_light_residual,
        normalized_weak_residual=normalized_weak_residual,
        normalized_light_weak_residual=normalized_light_weak_residual,
        normalized_clahe3_weak_residual=normalized_clahe3_weak_residual,
        bsgmf=bsgmf,
        top_hat=top_hat,
        frangi=frangi_map,
        hessian=hessian,
        combined=combined,
        triangle=triangle,
        triangle_clean=triangle_clean,
        final_binary=final_binary,
        od_mask=od_mask.astype(bool),
    )


def vessel_mask_quality_score(binary: np.ndarray, soft: np.ndarray, fov: np.ndarray) -> float:
    valid_area = max(int(np.count_nonzero(fov)), 1)
    density = float(np.count_nonzero(binary & fov) / valid_area)
    if density < 0.004 or density > 0.18:
        return -10.0

    support = float(soft[binary & fov].mean()) if np.any(binary & fov) else 0.0
    skel = skeletonize(binary & fov).astype(bool)
    skel_density = float(np.count_nonzero(skel) / valid_area)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(skel.astype(np.uint8), 8)
    areas = stats[1:, cv2.CC_STAT_AREA] if n_labels > 1 else np.array([], dtype=np.int32)
    component_count = int(areas.size)
    small_components = int(np.count_nonzero(areas < 8)) if areas.size else 0
    largest_fraction = float(areas.max() / max(int(np.count_nonzero(skel)), 1)) if areas.size else 0.0

    density_score = float(np.exp(-((density - 0.055) / 0.045) ** 2))
    length_score = min(skel_density / 0.018, 1.0)
    fragmentation_penalty = min(component_count / 120.0, 1.0) + min(small_components / 80.0, 1.0)
    dominance_penalty = max(0.0, largest_fraction - 0.55)

    return (
        0.45 * support
        + 0.35 * density_score
        + 0.25 * length_score
        - 0.18 * fragmentation_penalty
        - 0.25 * dominance_penalty
    )


def select_binary_mask_from_soft(
    soft: np.ndarray,
    fov: np.ndarray,
    config: VesselPipelineConfig,
    od_mask: np.ndarray | None = None,
) -> np.ndarray:
    candidates: list[tuple[float, np.ndarray]] = []
    base = replace(config, auto_binary_selection=False)

    threshold_methods: tuple[ThresholdMethod, ...] = ("hybrid", "percentile")
    densities = (0.10, 0.12, 0.14, 0.16, 0.18, 0.22, 0.26)
    erodes = (0, 8, 16)
    min_areas = (5, 15, 30)
    morphologies: tuple[MorphologyMode, ...] = ("none", "close")

    for threshold_method in threshold_methods:
        for density in densities:
            for erode_px in erodes:
                for min_area in min_areas:
                    for morphology in morphologies:
                        candidate_config = replace(
                            base,
                            threshold_method=threshold_method,
                            target_density=density,
                            fov_erode_px=erode_px,
                            min_component_area=min_area,
                            morphology=morphology,
                        )
                        candidate = postprocess_vessel_map(soft, fov, candidate_config, od_mask)
                        score = vessel_mask_quality_score(candidate, soft, fov)
                        candidates.append((score, candidate))

    if not candidates:
        return postprocess_vessel_map(soft, fov, base, od_mask)
    return max(candidates, key=lambda item: item[0])[1]


def postprocess_vessel_map(
    soft: np.ndarray,
    fov: np.ndarray,
    config: VesselPipelineConfig = VesselPipelineConfig(),
    od_mask: np.ndarray | None = None,
) -> np.ndarray:
    if config.auto_binary_selection:
        return select_binary_mask_from_soft(soft, fov, config, od_mask)

    inner_fov = erode_mask(fov, int(config.fov_erode_px))
    working = soft.copy()
    working[~inner_fov] = 0.0
    threshold = threshold_from_values(working[inner_fov], config.threshold_method, float(config.target_density), triangle_nbins=int(config.triangle_nbins))
    binary = (working >= threshold) & inner_fov

    binary = keep_components_at_least(binary, int(config.min_component_area))
    if config.hole_size > 0:
        binary = fill_holes_up_to(binary, int(config.hole_size))

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    if config.morphology in {"open", "open_close"}:
        binary = cv2.morphologyEx(binary.astype(np.uint8), cv2.MORPH_OPEN, kernel).astype(bool)
    if config.morphology in {"close", "open_close"}:
        binary = cv2.morphologyEx(binary.astype(np.uint8), cv2.MORPH_CLOSE, kernel).astype(bool)

    if config.od_suppression and od_mask is not None:
        binary = remove_optic_disc_components(binary, od_mask)
    binary = remove_border_components(binary, border_px=3)
    if config.final_skeletonize and binary.any():
        skel = skeletonize(binary).astype(bool)
        skel = keep_components_at_least(skel, int(config.skeleton_min_area))
        if config.skeleton_dilate_iter > 0 and skel.any():
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
            skel = cv2.dilate(skel.astype(np.uint8), kernel, iterations=int(config.skeleton_dilate_iter)).astype(bool)
        binary = skel
    return (binary & inner_fov).astype(bool)


def segment_vessels(rgb: np.ndarray, config: VesselPipelineConfig = VesselPipelineConfig()) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    features = compute_vessel_features(rgb, config)
    binary = postprocess_vessel_map(features.soft, features.fov, config, features.od_mask)
    return features.soft, binary, features.fov


def resize_binary_mask(mask: np.ndarray, size: int) -> np.ndarray:
    resized = cv2.resize(mask.astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)
    return resized.astype(bool)


def resize_soft_map(soft: np.ndarray, size: int) -> np.ndarray:
    resized = cv2.resize(soft.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
    return np.clip(resized, 0.0, 1.0).astype(np.float32)


def binary_mask_to_rgb(mask: np.ndarray) -> np.ndarray:
    image = mask.astype(np.uint8) * 255
    return np.repeat(image[:, :, None], 3, axis=2)


def vessel_guided_image(
    enhanced_rgb: np.ndarray,
    soft_or_binary: np.ndarray,
    fov: np.ndarray,
    guide_floor: float = 0.85,
    dilate_iter: int = 2,
    blur_sigma: float = 3.0,
) -> np.ndarray:
    if soft_or_binary.shape[:2] != enhanced_rgb.shape[:2]:
        if soft_or_binary.dtype == bool:
            guide_seed = resize_binary_mask(soft_or_binary, enhanced_rgb.shape[0]).astype(np.float32)
        else:
            guide_seed = resize_soft_map(soft_or_binary, enhanced_rgb.shape[0])
    else:
        guide_seed = soft_or_binary.astype(np.float32)
        if guide_seed.max() > 1:
            guide_seed /= 255.0

    if fov.shape[:2] != enhanced_rgb.shape[:2]:
        fov = resize_binary_mask(fov, enhanced_rgb.shape[0])
    if dilate_iter > 0:
        seed_u8 = np.clip(guide_seed * 255, 0, 255).astype(np.uint8)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        seed_u8 = cv2.dilate(seed_u8, kernel, iterations=int(dilate_iter))
        guide_seed = seed_u8.astype(np.float32) / 255.0
    blurred = cv2.GaussianBlur(guide_seed, (0, 0), sigmaX=blur_sigma, sigmaY=blur_sigma)
    guide = float(guide_floor) + (1.0 - float(guide_floor)) * normalize01(blurred, fov)
    guided = enhanced_rgb.astype(np.float32) * guide[:, :, None]
    guided[~fov] = 0
    return np.clip(guided, 0, 255).astype(np.uint8)

# Stable aliases used by the notebook wrapper functions.
inlined_classical_segment_vessels = segment_vessels
inlined_classical_binary_mask_to_rgb = binary_mask_to_rgb
inlined_classical_resize_binary_mask = resize_binary_mask
inlined_classical_resize_soft_map = resize_soft_map
inlined_classical_vessel_guided_image = vessel_guided_image


In [ ]:
sample_rows = split_df.groupby('label', group_keys=False).sample(n=1, random_state=cfg.seed)
images = [resize_rgb(read_rgb(row.path), cfg.image_size) for row in sample_rows.itertuples()]
titles = [row.label for row in sample_rows.itertuples()]
show_image_grid(images, titles, cols=4, figsize=(12, 4))

## Deterministic Image Processing

S2 is a conservative Rahim-inspired enhancement. S3 and S4 are based on an Almeida-inspired vessel pipeline:

1. estimate field-of-view mask;
2. use green channel;
3. apply CLAHE and background normalization;
4. compute vesselness from Frangi and top-hat features;
5. threshold the soft map with the configured method;
6. clean binary mask with morphology;
7. for S4, convert the cleaned binary mask into a blurred soft guide and multiply it with the enhanced RGB image.

In [ ]:
def enhance_for_classification(rgb: np.ndarray) -> np.ndarray:
    rgb = resize_rgb(rgb, cfg.image_size)
    fov = estimate_fov_mask(rgb)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l_channel)
    enhanced = cv2.merge([l_enhanced, a_channel, b_channel])
    enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2RGB)
    enhanced[~fov] = 0
    return enhanced


def build_vessel_config() -> VesselPipelineConfig:
    return VesselPipelineConfig(
        process_max_side=cfg.vessel_process_max_side,
        vesselness_mode=cfg.vesselness_mode,
        clahe_clip=2.5,
        background_sigma=30.0,
        threshold_method=cfg.vessel_threshold_method,
        target_density=cfg.vessel_target_density,
        triangle_nbins=cfg.vessel_triangle_nbins,
        fov_erode_px=cfg.vessel_fov_erode_px,
        min_component_area=cfg.vessel_min_size,
        hole_size=cfg.vessel_hole_size,
        morphology=cfg.vessel_morphology,
        od_suppression=cfg.vessel_od_suppression,
        od_soft_penalty=cfg.vessel_od_soft_penalty,
        final_skeletonize=cfg.vessel_final_skeletonize,
        skeleton_min_area=cfg.vessel_skeleton_min_area,
        skeleton_dilate_iter=cfg.vessel_skeleton_dilate_iter,
        auto_binary_selection=cfg.vessel_auto_binary_selection,
        top_hat_weight=cfg.vessel_top_hat_weight,
        matched_weight=cfg.vessel_matched_weight,
        frangi_weight=cfg.vessel_frangi_weight,
        jerman_weight=cfg.vessel_jerman_weight,
    )


def segment_vessels(rgb: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    return inlined_classical_segment_vessels(rgb, build_vessel_config())


def binary_mask_to_rgb(mask: np.ndarray) -> np.ndarray:
    return inlined_classical_binary_mask_to_rgb(mask)


def resize_binary_mask(mask: np.ndarray, size: int = cfg.image_size) -> np.ndarray:
    return inlined_classical_resize_binary_mask(mask, size)


def resize_soft_map(soft: np.ndarray, size: int = cfg.image_size) -> np.ndarray:
    return inlined_classical_resize_soft_map(soft, size)


def vessel_guided_image(enhanced_rgb: np.ndarray, soft_or_binary: np.ndarray, fov: np.ndarray) -> np.ndarray:
    return inlined_classical_vessel_guided_image(
        enhanced_rgb,
        soft_or_binary,
        fov,
        guide_floor=cfg.vessel_guide_floor,
        dilate_iter=cfg.vessel_guide_dilate_iter,
        blur_sigma=cfg.vessel_guide_blur_sigma,
    )


In [ ]:
row = sample_rows.iloc[0]
original = read_rgb(row['path'])
raw = resize_rgb(original, cfg.image_size)
enhanced = enhance_for_classification(original)
soft, binary, fov = segment_vessels(original)
soft_224 = resize_soft_map(soft, cfg.image_size)
binary_224 = resize_binary_mask(binary, cfg.image_size)
fov_224 = resize_binary_mask(fov, cfg.image_size)
vessel_only = binary_mask_to_rgb((soft_224 > 0).astype(bool))
vessel_only = np.repeat((soft_224 * 255).astype(np.uint8)[:, :, None], 3, axis=2)
guided = vessel_guided_image(enhanced, soft_224, fov_224)
show_image_grid(
    [raw, enhanced, (soft_224 * 255).astype(np.uint8), binary_mask_to_rgb(binary_224), vessel_only, guided],
    ['S1 raw', 'S2 enhanced', 'soft vesselness', 'binary diagnostic', 'S3 vesselness only', 'S4 vessel-guided'],
    cols=6,
    figsize=(16, 4),
)

## Vessel Segmentation Evaluation on Agrawal2021

Agrawal2021 is used only to check whether the binary vessel masks are reasonable. The ground truth masks are treated as binary masks. If a mask file has antialiasing or grayscale edges, it is binarized before metric calculation.

In [ ]:
def find_agrawal_pairs(root: Optional[Path]) -> pd.DataFrame:
    if root is None:
        return pd.DataFrame(columns=['image_path', 'mask_path', 'subset'])
    bv_root = root / 'HVDROPDB-BV'
    candidates = [
        ('RetCam', bv_root / 'RetCam_Vessels_images', bv_root / 'RetCam_Vessels_masks'),
        ('RetCam', bv_root / 'Retcam_Vessels_images', bv_root / 'Retcam_Vessels_masks'),
        ('Neo', bv_root / 'Neo_Vessels_images', bv_root / 'Neo_Vessels_masks'),
    ]
    rows = []
    for subset, image_dir, mask_dir in candidates:
        if not image_dir.is_dir() or not mask_dir.is_dir():
            continue
        images = {p.stem: p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS}
        masks = {p.stem: p for p in mask_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS}
        for stem in sorted(set(images) & set(masks)):
            rows.append({'image_path': str(images[stem]), 'mask_path': str(masks[stem]), 'subset': subset})
    return pd.DataFrame(rows)


def read_binary_mask(path: Path | str, target_shape: Tuple[int, int]) -> np.ndarray:
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f'Could not read mask: {path}')
    height, width = target_shape
    mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
    if mask.max() <= 1:
        return mask.astype(bool)
    return mask >= 127


def segmentation_metrics(pred: np.ndarray, gt: np.ndarray) -> Dict[str, float]:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    tn = np.logical_and(~pred, ~gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    eps = 1e-8
    dice = (2 * tp) / (2 * tp + fp + fn + eps)
    iou = tp / (tp + fp + fn + eps)
    sensitivity = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)
    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    return {
        'dice': float(dice),
        'iou': float(iou),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'accuracy': float(accuracy),
    }


def evaluate_agrawal(max_images: Optional[int] = None) -> pd.DataFrame:
    pairs = find_agrawal_pairs(AGRAWAL_ROOT)
    if pairs.empty:
        print('Agrawal2021 vessel pairs were not found. Skipping vessel segmentation evaluation.')
        return pd.DataFrame()
    if max_images is not None:
        pairs = pairs.head(max_images)
    rows = []
    for row in tqdm(pairs.itertuples(index=False), total=len(pairs), desc='Agrawal vessel eval'):
        image = read_rgb(row.image_path)
        _, pred_binary, _ = segment_vessels(image)
        gt_binary = read_binary_mask(row.mask_path, pred_binary.shape)
        metrics = segmentation_metrics(pred_binary, gt_binary)
        metrics.update({'image_path': row.image_path, 'mask_path': row.mask_path, 'subset': row.subset})
        rows.append(metrics)
    metrics_df = pd.DataFrame(rows)
    metrics_path = OUTPUT_RESULTS_DIR / 'agrawal_vessel_segmentation_metrics.csv'
    metrics_df.to_csv(metrics_path, index=False)
    print(f'Saved vessel segmentation metrics: {metrics_path}')
    return metrics_df

if cfg.run_segmentation_eval:
    agrawal_metrics_df = evaluate_agrawal()
    if not agrawal_metrics_df.empty:
        display(agrawal_metrics_df.groupby('subset')[['dice', 'iou', 'sensitivity', 'specificity', 'accuracy']].mean())
else:
    agrawal_metrics_df = pd.DataFrame()

## DEBUG MODE: Fixed Baseline Visualization

When `cfg.debug_mode` is enabled, this cell generates the canonical 15-image per-process vessel visualization and summary artifacts under `output/00_debug_baseline/normalization_residual_tuned/`.


In [ ]:
def numeric_key(path: Path) -> Tuple[int, str]:
    try:
        return int(path.stem), path.stem
    except ValueError:
        return 10**9, path.stem


def find_agrawal_debug_subset(root: Optional[Path], subset: str, limit: int = 5) -> List[Dict[str, object]]:
    if root is None:
        return []
    bv_root = root / 'HVDROPDB-BV'
    if subset == 'RetCam':
        candidates = [
            (bv_root / 'RetCam_Vessels_images', bv_root / 'RetCam_Vessels_masks'),
            (bv_root / 'Retcam_Vessels_images', bv_root / 'Retcam_Vessels_masks'),
        ]
    elif subset == 'Neo':
        candidates = [(bv_root / 'Neo_Vessels_images', bv_root / 'Neo_Vessels_masks')]
    else:
        raise ValueError(f'Unknown Agrawal subset: {subset}')

    for image_dir, mask_dir in candidates:
        if not image_dir.is_dir() or not mask_dir.is_dir():
            continue
        images = {path.stem: path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS}
        masks = {path.stem: path for path in mask_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS}
        rows = []
        for stem in sorted(set(images) & set(masks), key=lambda value: numeric_key(Path(value))):
            rows.append({
                'source': subset,
                'label': subset,
                'image_path': images[stem],
                'mask_path': masks[stem],
                'name': images[stem].name,
            })
        return rows[:limit]
    return []


def find_zhao_debug_samples(root: Path, limit: int = 5) -> List[Dict[str, object]]:
    classes = [path for path in sorted(root.iterdir()) if path.is_dir() and path.name.lower() not in cfg.excluded_classes]
    class_files = {
        klass.name: sorted([path for path in klass.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS])
        for klass in classes
    }
    rows = []
    round_idx = 0
    while len(rows) < limit:
        added = False
        for class_name, files in class_files.items():
            if round_idx < len(files):
                path = files[round_idx]
                rows.append({
                    'source': 'Zhao',
                    'label': class_name,
                    'image_path': path,
                    'mask_path': None,
                    'name': path.name,
                })
                added = True
                if len(rows) >= limit:
                    break
        if not added:
            break
        round_idx += 1
    return rows


def gray_rgb(image: np.ndarray) -> np.ndarray:
    if image.dtype == bool:
        image = image.astype(np.uint8) * 255
    elif image.dtype != np.uint8:
        image = np.clip(image.astype(np.float32) * 255.0, 0, 255).astype(np.uint8)
    return np.repeat(image[:, :, None], 3, axis=2)


def resize_tile(image: np.ndarray, size: int) -> np.ndarray:
    if image.ndim == 2:
        image = gray_rgb(image)
    return cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)


def add_tile_label(image: np.ndarray, label: str, size: int) -> np.ndarray:
    output = resize_tile(image, size)
    cv2.rectangle(output, (0, 0), (size, 24), (0, 0, 0), -1)
    cv2.putText(output, label[:34], (4, 17), cv2.FONT_HERSHEY_SIMPLEX, 0.43, (255, 255, 255), 1, cv2.LINE_AA)
    return output


def overlay_prediction(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    overlay = np.zeros((*gt.shape, 3), dtype=np.uint8)
    overlay[np.logical_and(pred, gt)] = (255, 255, 255)
    overlay[np.logical_and(pred, ~gt)] = (255, 0, 0)
    overlay[np.logical_and(~pred, gt)] = (0, 255, 255)
    return overlay


def empty_tile(shape: Tuple[int, int]) -> np.ndarray:
    return np.zeros((*shape, 3), dtype=np.uint8)


def debug_binary_metrics(pred: np.ndarray, gt: np.ndarray) -> Dict[str, float]:
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    tp = int(np.logical_and(pred, gt).sum())
    tn = int(np.logical_and(~pred, ~gt).sum())
    fp = int(np.logical_and(pred, ~gt).sum())
    fn = int(np.logical_and(~pred, gt).sum())
    eps = 1e-9
    return {
        'dice': float((2 * tp) / (2 * tp + fp + fn + eps)),
        'iou': float(tp / (tp + fp + fn + eps)),
        'precision': float(tp / (tp + fp + eps)),
        'sensitivity': float(tp / (tp + fn + eps)),
        'specificity': float(tn / (tn + fp + eps)),
        'pred_density': float(pred.mean()),
        'gt_density': float(gt.mean()),
    }


def debug_soft_metrics(score: np.ndarray, gt: np.ndarray) -> Dict[str, float]:
    y_true = gt.reshape(-1).astype(np.uint8)
    y_score = score.reshape(-1).astype(np.float32)
    metrics = {'auprc': float(average_precision_score(y_true, y_score))}
    metrics['roc_auc'] = float(roc_auc_score(y_true, y_score)) if np.unique(y_true).size == 2 else float('nan')
    return metrics


def write_rows_csv(path: Path, rows: Iterable[Dict[str, object]]) -> None:
    rows = list(rows)
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def vessel_fusion_weight_candidates() -> Dict[str, Tuple[float, float, float, float]]:
    return {
        'current_almeida': (0.20, 0.40, 0.15, 0.25),
        'tuned_hessian_frangi': (0.05, 0.25, 0.20, 0.50),
        'matched_jerman': (0.08, 0.52, 0.05, 0.35),
        'matched_heavy': (0.05, 0.65, 0.05, 0.25),
        'no_frangi': (0.10, 0.55, 0.00, 0.35),
    }


def run_vessel_fusion_tuning(rows: List[Dict[str, object]], output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    base_config = replace(build_vessel_config(), vesselness_mode="almeida")
    threshold_candidates = [
        ('triangle', 0.14, 12, 4),
        ('percentile', 0.08, 12, 8),
        ('percentile', 0.10, 12, 8),
        ('percentile', 0.12, 12, 8),
        ('hybrid', 0.10, 12, 8),
    ]

    cached_maps = []
    for row in tqdm(rows, desc='DEBUG vessel tuning maps'):
        if row['mask_path'] is None:
            continue
        rgb = read_rgb(row['image_path'])
        working_rgb = resize_max_side(rgb, base_config.process_max_side)
        fov = estimate_fov_mask(working_rgb)
        *_, enhanced_green = preprocess_green_channel(working_rgb, fov, base_config)
        inverted = 255 - enhanced_green
        inverted_float = normalize01(inverted.astype(np.float32), fov)
        cached_maps.append({
            'source': row['source'],
            'image': row['name'],
            'fov': fov,
            'gt': read_binary_mask(row['mask_path'], fov.shape),
            'top_hat': modified_tophat(inverted, fov),
            'matched': matched_filter_response(inverted_float, fov),
            'frangi': normalize01(safe_filter_call('frangi', inverted_float), fov),
            'jerman': jerman_vesselness(inverted_float, fov),
        })

    tuning_rows = []
    for candidate_name, weights in vessel_fusion_weight_candidates().items():
        top_hat_weight, matched_weight, frangi_weight, jerman_weight = weights
        candidate_config = replace(
            base_config,
            top_hat_weight=top_hat_weight,
            matched_weight=matched_weight,
            frangi_weight=frangi_weight,
            jerman_weight=jerman_weight,
        )
        for threshold_method, target_density, fov_erode_px, min_component_area in threshold_candidates:
            metric_rows = []
            post_config = replace(
                candidate_config,
                threshold_method=threshold_method,
                target_density=target_density,
                fov_erode_px=fov_erode_px,
                min_component_area=min_component_area,
            )
            for item in cached_maps:
                soft = fuse_vessel_responses(
                    item['top_hat'],
                    item['matched'],
                    item['frangi'],
                    item['jerman'],
                    item['fov'],
                    mode=post_config.vesselness_mode,
                    config=post_config,
                )
                pred = postprocess_vessel_map(soft, item['fov'], post_config)
                metric_rows.append(debug_binary_metrics(pred, item['gt']))
            if not metric_rows:
                continue
            metric_mean = {key: float(np.mean([row[key] for row in metric_rows])) for key in metric_rows[0]}
            tuning_rows.append({
                'candidate': candidate_name,
                'top_hat_weight': top_hat_weight,
                'matched_weight': matched_weight,
                'frangi_weight': frangi_weight,
                'jerman_weight': jerman_weight,
                'threshold_method': threshold_method,
                'target_density': target_density,
                'fov_erode_px': fov_erode_px,
                'min_component_area': min_component_area,
                **metric_mean,
            })

    tuning_rows = sorted(tuning_rows, key=lambda row: (row['dice'], row['iou']), reverse=True)
    tuning_path = output_dir / 'debug_vessel_tuning_summary.csv'
    write_rows_csv(tuning_path, tuning_rows)
    return tuning_path


def local_clahe_channel(
    channel: np.ndarray,
    fov: np.ndarray,
    clip_limit: float = 2.2,
    tile_grid_size: tuple[int, int] = (16, 16),
) -> np.ndarray:
    output = np.clip(channel, 0, 255).astype(np.uint8)
    output = cv2.createCLAHE(clipLimit=float(clip_limit), tileGridSize=tile_grid_size).apply(output)
    output[~fov] = 0
    return output


def small_blackhat_boost(channel: np.ndarray, fov: np.ndarray, strength: float = 0.75) -> np.ndarray:
    source = np.clip(channel, 0, 255).astype(np.uint8)
    responses = []
    for kernel_size in (5, 7, 9, 11):
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        responses.append(cv2.morphologyEx(source, cv2.MORPH_BLACKHAT, kernel).astype(np.float32))
    blackhat = np.max(np.stack(responses, axis=0), axis=0)
    boosted = source.astype(np.float32) - float(strength) * blackhat
    boosted[~fov] = 0
    return np.clip(boosted, 0, 255).astype(np.uint8)


def fixed_center_uint8(values: np.ndarray, center: float = 0.0, scale: float = 1.0) -> np.ndarray:
    display = 128.0 + (values.astype(np.float32) - float(center)) * float(scale)
    return np.clip(display, 0, 255).astype(np.uint8)


def fixed_positive_uint8(values: np.ndarray, scale: float = 1.0) -> np.ndarray:
    display = values.astype(np.float32) * float(scale)
    return np.clip(display, 0, 255).astype(np.uint8)


def local_mean_difference_channel(channel: np.ndarray, fov: np.ndarray, kernel_size: int = 31) -> np.ndarray:
    source = np.clip(channel, 0, 255).astype(np.uint8)
    filled = source.astype(np.float32).copy()
    if np.any(fov):
        filled[~fov] = float(np.median(filled[fov]))
    background = cv2.blur(filled, (int(kernel_size), int(kernel_size)), borderType=cv2.BORDER_REFLECT)
    deficit = background - source.astype(np.float32)
    deficit[~fov] = 0
    return fixed_positive_uint8(deficit, scale=3.0)


def local_blackhat_channel(channel: np.ndarray, fov: np.ndarray, kernel_size: int = 15, scale: float = 2.0) -> np.ndarray:
    source = np.clip(channel, 0, 255).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(kernel_size), int(kernel_size)))
    blackhat = cv2.morphologyEx(source, cv2.MORPH_BLACKHAT, kernel).astype(np.float32)
    blackhat[~fov] = 0
    return fixed_positive_uint8(blackhat, scale=scale)


def debug_raw_channel_candidates(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> list[tuple[str, np.ndarray]]:
    rgb = np.clip(rgb, 0, 255).astype(np.uint8)
    red = rgb[:, :, 0]
    green = rgb[:, :, 1]
    blue = rgb[:, :, 2]

    lab_raw = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lab_l, lab_a, lab_b = cv2.split(lab_raw)
    cielab_rgb, cielab_l = enhance_cielab_rgb(rgb, fov, config)
    cielab_green = cielab_rgb[:, :, 1]
    cielab_lab = cv2.cvtColor(cielab_rgb, cv2.COLOR_RGB2LAB)
    cielab_l2, cielab_a2, cielab_b2 = cv2.split(cielab_lab)
    cielab4_rgb, cielab4_l = enhance_cielab_rgb(rgb, fov, replace(config, clahe_clip=4.0))
    cielab4_green = cielab4_rgb[:, :, 1]
    cielab6_rgb, cielab6_l = enhance_cielab_rgb(rgb, fov, replace(config, clahe_clip=6.0))
    cielab6_green = cielab6_rgb[:, :, 1]

    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hsv_h, hsv_s, hsv_v = cv2.split(hsv)
    ycrcb = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)
    y_chan, cr_chan, cb_chan = cv2.split(ycrcb)

    red_float = red.astype(np.float32) + 1.0
    green_float = green.astype(np.float32) + 1.0
    blue_float = blue.astype(np.float32) + 1.0
    log_rg = fixed_center_uint8(np.log(red_float / green_float), center=0.0, scale=80.0)
    log_gb = fixed_center_uint8(np.log(green_float / blue_float), center=0.0, scale=80.0)
    red_minus_green = fixed_center_uint8(red.astype(np.float32) - green.astype(np.float32), center=0.0, scale=0.5)
    green_minus_red = fixed_center_uint8(green.astype(np.float32) - red.astype(np.float32), center=0.0, scale=0.5)
    excess_green = fixed_center_uint8(2.0 * green.astype(np.float32) - red.astype(np.float32) - blue.astype(np.float32), center=0.0, scale=0.35)

    candidates = [
        ('Raw R', red),
        ('Raw G', green),
        ('Raw B', blue),
        ('Raw LAB L', lab_l),
        ('Raw LAB a', lab_a),
        ('Raw LAB b', lab_b),
        ('CIELAB L*', cielab_l),
        ('CIELAB green', cielab_green),
        ('CIELAB4 L*', cielab4_l),
        ('CIELAB4 green', cielab4_green),
        ('CIELAB6 L*', cielab6_l),
        ('CIELAB6 green', cielab6_green),
        ('CIELAB LAB L', cielab_l2),
        ('CIELAB LAB a', cielab_a2),
        ('CIELAB LAB b', cielab_b2),
        ('HSV H', hsv_h),
        ('HSV S', hsv_s),
        ('HSV V', hsv_v),
        ('YCrCb Y', y_chan),
        ('YCrCb Cr', cr_chan),
        ('YCrCb Cb', cb_chan),
        ('R-G fixed', red_minus_green),
        ('G-R fixed', green_minus_red),
        ('log R/G fixed', log_rg),
        ('log G/B fixed', log_gb),
        ('Excess G fixed', excess_green),
        ('Raw G CLAHE8', local_clahe_channel(green, fov, clip_limit=2.5, tile_grid_size=(8, 8))),
        ('Raw G CLAHE16', local_clahe_channel(green, fov, clip_limit=2.2, tile_grid_size=(16, 16))),
        ('CIELAB G CLAHE8', local_clahe_channel(cielab_green, fov, clip_limit=2.5, tile_grid_size=(8, 8))),
        ('CIELAB G CLAHE16', local_clahe_channel(cielab_green, fov, clip_limit=2.2, tile_grid_size=(16, 16))),
        ('CIELAB G CLAHE4', local_clahe_channel(cielab_green, fov, clip_limit=4.0, tile_grid_size=(16, 16))),
        ('CIELAB G CLAHE6', local_clahe_channel(cielab_green, fov, clip_limit=6.0, tile_grid_size=(16, 16))),
        ('CIELAB4 G CLAHE4', local_clahe_channel(cielab4_green, fov, clip_limit=4.0, tile_grid_size=(16, 16))),
        ('CIELAB6 G CLAHE6', local_clahe_channel(cielab6_green, fov, clip_limit=6.0, tile_grid_size=(16, 16))),
        ('Raw G local deficit', local_mean_difference_channel(green, fov, kernel_size=31)),
        ('CIELAB G local deficit', local_mean_difference_channel(cielab_green, fov, kernel_size=31)),
        ('Raw G blackhat', local_blackhat_channel(green, fov, kernel_size=15, scale=2.0)),
        ('CIELAB G blackhat', local_blackhat_channel(cielab_green, fov, kernel_size=15, scale=2.0)),
        ('LAB L local deficit', local_mean_difference_channel(lab_l, fov, kernel_size=31)),
        ('CIELAB L local deficit', local_mean_difference_channel(cielab_l, fov, kernel_size=31)),
    ]
    output = []
    for label, channel in candidates:
        display = np.clip(channel, 0, 255).astype(np.uint8).copy()
        display[~fov] = 0
        output.append((label, display))
    return output


def run_debug_raw_channel_visualization(rows: List[Dict[str, object]], output_dir: Path, tile_size: int = 150) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    config = build_vessel_config()
    sheet_rows = []
    for idx, row in enumerate(tqdm(rows, desc='DEBUG raw channel visualization'), start=1):
        rgb = read_rgb(row['image_path'])
        working_rgb = resize_max_side(rgb, config.process_max_side)
        fov = estimate_fov_mask(working_rgb)
        gt = read_binary_mask(row['mask_path'], fov.shape) if row.get('mask_path') is not None else None
        gt_tile = gt.astype(np.uint8) * 255 if gt is not None else empty_tile(fov.shape)
        title = f'{idx}. {row["source"]} {row["name"]}'
        row_tiles = [
            add_tile_label(working_rgb, title, tile_size),
            add_tile_label(fov.astype(np.uint8) * 255, 'FOV', tile_size),
            add_tile_label(gt_tile, 'GT', tile_size),
        ]
        for label, channel in debug_raw_channel_candidates(working_rgb, fov, config):
            row_tiles.append(add_tile_label(channel, label, tile_size))
        sheet_rows.append(row_tiles)
    sheet = np.concatenate([np.concatenate(row_tiles, axis=1) for row_tiles in sheet_rows], axis=0)
    sheet_path = output_dir / 'debug_raw_channel_visualization_15_samples.jpg'
    cv2.imwrite(str(sheet_path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    return {'sheet': sheet_path}


def cielab_green_clahe_source(
    rgb: np.ndarray,
    fov: np.ndarray,
    config: VesselPipelineConfig,
    l_clip: float = 6.0,
    green_clip: float = 6.0,
    tile_grid_size: tuple[int, int] = (16, 16),
) -> np.ndarray:
    enhanced_rgb, _ = enhance_cielab_rgb(rgb, fov, replace(config, clahe_clip=float(l_clip)))
    green = enhanced_rgb[:, :, 1]
    return local_clahe_channel(green, fov, clip_limit=float(green_clip), tile_grid_size=tile_grid_size)


def masked_median_value(channel: np.ndarray, fov: np.ndarray) -> float:
    return float(np.median(channel[fov])) if np.any(fov) else 0.0


def fill_outside_fov(channel: np.ndarray, fov: np.ndarray) -> np.ndarray:
    filled = channel.astype(np.float32).copy()
    if np.any(fov):
        filled[~fov] = masked_median_value(channel, fov)
    return filled


def gaussian_background_flatten(channel: np.ndarray, fov: np.ndarray, sigma: float = 30.0) -> np.ndarray:
    filled = fill_outside_fov(channel, fov)
    background = cv2.GaussianBlur(filled, (0, 0), sigmaX=float(sigma), sigmaY=float(sigma), borderType=cv2.BORDER_REFLECT)
    corrected = filled - background + masked_median_value(background.astype(np.uint8), fov)
    corrected[~fov] = 0
    return np.clip(corrected, 0, 255).astype(np.uint8)


def divide_background_flatten(channel: np.ndarray, fov: np.ndarray, sigma: float = 30.0) -> np.ndarray:
    filled = fill_outside_fov(channel, fov)
    background = cv2.GaussianBlur(filled, (0, 0), sigmaX=float(sigma), sigmaY=float(sigma), borderType=cv2.BORDER_REFLECT)
    scale = float(np.median(background[fov])) if np.any(fov) else 1.0
    corrected = filled * scale / np.maximum(background, 1.0)
    corrected[~fov] = 0
    return np.clip(corrected, 0, 255).astype(np.uint8)


def box_background_flatten(channel: np.ndarray, fov: np.ndarray, kernel_size: int = 51) -> np.ndarray:
    filled = fill_outside_fov(channel, fov)
    kernel_size = max(3, int(kernel_size))
    background = cv2.blur(filled, (kernel_size, kernel_size), borderType=cv2.BORDER_REFLECT)
    corrected = filled - background + (float(np.median(background[fov])) if np.any(fov) else 0.0)
    corrected[~fov] = 0
    return np.clip(corrected, 0, 255).astype(np.uint8)


def morph_background_flatten(channel: np.ndarray, fov: np.ndarray, kernel_size: int = 31) -> np.ndarray:
    source = np.clip(fill_outside_fov(channel, fov), 0, 255).astype(np.uint8)
    kernel_size = max(3, int(kernel_size) | 1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    background = cv2.morphologyEx(source, cv2.MORPH_CLOSE, kernel)
    corrected = source.astype(np.float32) - background.astype(np.float32) + masked_median_value(background, fov)
    corrected[~fov] = 0
    return np.clip(corrected, 0, 255).astype(np.uint8)


def cielab6_green_pre_clahe(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> np.ndarray:
    enhanced_rgb, _ = enhance_cielab_rgb(rgb, fov, replace(config, clahe_clip=6.0))
    green = enhanced_rgb[:, :, 1].copy()
    green[~fov] = 0
    return green.astype(np.uint8)


def multiscale_blackhat_response(
    channel: np.ndarray,
    fov: np.ndarray,
    kernel_sizes: tuple[int, ...] = (7, 11, 15, 21, 31),
) -> np.ndarray:
    source = np.clip(fill_outside_fov(channel, fov), 0, 255).astype(np.uint8)
    responses = []
    for kernel_size in kernel_sizes:
        kernel_size = max(3, int(kernel_size) | 1)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        responses.append(cv2.morphologyEx(source, cv2.MORPH_BLACKHAT, kernel).astype(np.float32))
    response = np.max(np.stack(responses, axis=0), axis=0)
    response[~fov] = 0
    return response.astype(np.float32)


def blackhat_boost_then_clahe_source(
    channel: np.ndarray,
    fov: np.ndarray,
    strength: float = 1.25,
    green_clip: float = 6.0,
) -> np.ndarray:
    blackhat = multiscale_blackhat_response(channel, fov)
    boosted = channel.astype(np.float32) - float(strength) * blackhat
    boosted[~fov] = 0
    boosted = np.clip(boosted, 0, 255).astype(np.uint8)
    return local_clahe_channel(boosted, fov, clip_limit=float(green_clip), tile_grid_size=(16, 16))


def blackhat_isolated_dark_source(channel: np.ndarray, fov: np.ndarray) -> np.ndarray:
    blackhat = multiscale_blackhat_response(channel, fov)
    response = normalize01(blackhat, fov)
    source = np.zeros(channel.shape, dtype=np.float32)
    source[fov] = 220.0 - 205.0 * response[fov]
    return np.clip(source, 0, 255).astype(np.uint8)


def debug_flatten_darkline_filter_sources(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> list[tuple[str, np.ndarray]]:
    c6_green = cielab6_green_pre_clahe(rgb, fov, config)
    flat_sub = gaussian_background_flatten(c6_green, fov, sigma=30.0)
    flat_div = divide_background_flatten(c6_green, fov, sigma=30.0)
    return [
        ('C6G6', local_clahe_channel(c6_green, fov, clip_limit=6.0, tile_grid_size=(16, 16))),
        ('FlatSub+G6', local_clahe_channel(flat_sub, fov, clip_limit=6.0, tile_grid_size=(16, 16))),
        ('FlatDiv+G6', local_clahe_channel(flat_div, fov, clip_limit=6.0, tile_grid_size=(16, 16))),
        ('BHboost+G6', blackhat_boost_then_clahe_source(c6_green, fov, strength=1.25, green_clip=6.0)),
        ('BH isolated', blackhat_isolated_dark_source(c6_green, fov)),
    ]


def bilateral_condition(channel: np.ndarray, fov: np.ndarray, d: int, sigma_color: float, sigma_space: float) -> np.ndarray:
    source = np.clip(fill_outside_fov(channel, fov), 0, 255).astype(np.uint8)
    if d % 2 == 0:
        d += 1
    output = cv2.bilateralFilter(source, d=int(d), sigmaColor=float(sigma_color), sigmaSpace=float(sigma_space))
    output[~fov] = 0
    return output.astype(np.uint8)


def nlm_condition(channel: np.ndarray, fov: np.ndarray, h: float = 7.0) -> np.ndarray:
    source = np.clip(fill_outside_fov(channel, fov), 0, 255).astype(np.uint8)
    output = cv2.fastNlMeansDenoising(source, None, h=float(h), templateWindowSize=7, searchWindowSize=21)
    output[~fov] = 0
    return output.astype(np.uint8)


def local_texture_capped_blend(strong: np.ndarray, conservative: np.ndarray, fov: np.ndarray) -> np.ndarray:
    filled = fill_outside_fov(strong, fov)
    mean = cv2.blur(filled, (15, 15), borderType=cv2.BORDER_REFLECT)
    mean_sq = cv2.blur(filled * filled, (15, 15), borderType=cv2.BORDER_REFLECT)
    local_std = np.sqrt(np.maximum(mean_sq - mean * mean, 0.0))
    alpha = np.clip((local_std - 10.0) / 30.0, 0.0, 1.0)
    output = strong.astype(np.float32) * (1.0 - 0.65 * alpha) + conservative.astype(np.float32) * (0.65 * alpha)
    output[~fov] = 0
    return np.clip(output, 0, 255).astype(np.uint8)


def bright_artifact_suppress(channel: np.ndarray, rgb: np.ndarray, fov: np.ndarray, sigma: float = 18.0) -> np.ndarray:
    source = np.clip(fill_outside_fov(channel, fov), 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l_channel = lab[:, :, 0]
    if np.any(fov):
        l_cut = np.percentile(l_channel[fov], 98.0)
        s_cut = np.percentile(source[fov], 95.0)
    else:
        l_cut, s_cut = 255, 255
    artifact = (l_channel >= l_cut) & (source >= s_cut) & fov
    artifact = cv2.dilate(artifact.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)), iterations=1).astype(bool)
    replacement = cv2.GaussianBlur(source.astype(np.float32), (0, 0), sigmaX=float(sigma), sigmaY=float(sigma), borderType=cv2.BORDER_REFLECT)
    output = source.astype(np.float32)
    output[artifact] = replacement[artifact]
    output[~fov] = 0
    return np.clip(output, 0, 255).astype(np.uint8)


def fov_rim_quiet(channel: np.ndarray, fov: np.ndarray, rim_px: int = 18) -> np.ndarray:
    output = np.clip(channel, 0, 255).astype(np.uint8).copy()
    inner = erode_mask(fov, int(rim_px))
    if np.any(fov):
        output[fov & ~inner] = int(round(masked_median_value(output, inner if np.any(inner) else fov)))
    output[~fov] = 0
    return output


def debug_conditioning_candidates(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> list[tuple[str, np.ndarray]]:
    c6 = cielab_green_clahe_source(rgb, fov, config, l_clip=6.0, green_clip=6.0)
    c4 = cielab_green_clahe_source(rgb, fov, config, l_clip=4.0, green_clip=4.0)
    blend_70 = np.clip(0.70 * c6.astype(np.float32) + 0.30 * c4.astype(np.float32), 0, 255).astype(np.uint8)
    bilat_mild = bilateral_condition(c6, fov, d=5, sigma_color=25.0, sigma_space=5.0)
    bg_gauss = gaussian_background_flatten(c6, fov, sigma=30.0)
    combined = gaussian_background_flatten(bilat_mild, fov, sigma=30.0)
    combined = np.clip(0.70 * combined.astype(np.float32) + 0.30 * c4.astype(np.float32), 0, 255).astype(np.uint8)
    combined = fov_rim_quiet(combined, fov, rim_px=18)
    return [
        ('C6G6 source', c6),
        ('C4G4 source', c4),
        ('Blend 70C6 30C4', blend_70),
        ('Gauss bg sub s20', gaussian_background_flatten(c6, fov, sigma=20.0)),
        ('Gauss bg sub s30', bg_gauss),
        ('Gauss divide s30', divide_background_flatten(c6, fov, sigma=30.0)),
        ('Box bg sub 51', box_background_flatten(c6, fov, kernel_size=51)),
        ('Morph close bg 31', morph_background_flatten(c6, fov, kernel_size=31)),
        ('Bilateral mild', bilat_mild),
        ('Bilateral strong', bilateral_condition(c6, fov, d=9, sigma_color=45.0, sigma_space=9.0)),
        ('NLM mild', nlm_condition(c6, fov, h=7.0)),
        ('Median3 then bg', gaussian_background_flatten(cv2.medianBlur(c6, 3), fov, sigma=30.0)),
        ('Texture capped blend', local_texture_capped_blend(c6, c4, fov)),
        ('Bright artifact suppress', bright_artifact_suppress(c6, rgb, fov)),
        ('FOV rim quiet', fov_rim_quiet(c6, fov, rim_px=18)),
        ('Combined clean candidate', combined),
    ]


def run_debug_conditioning_visualization(rows: List[Dict[str, object]], output_dir: Path, tile_size: int = 150) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    config = build_vessel_config()
    sheet_rows = []
    for idx, row in enumerate(tqdm(rows, desc='DEBUG conditioning visualization'), start=1):
        rgb = read_rgb(row['image_path'])
        working_rgb = resize_max_side(rgb, config.process_max_side)
        fov = estimate_fov_mask(working_rgb)
        gt = read_binary_mask(row['mask_path'], fov.shape) if row.get('mask_path') is not None else None
        gt_tile = gt.astype(np.uint8) * 255 if gt is not None else empty_tile(fov.shape)
        title = f'{idx}. {row["source"]} {row["name"]}'
        row_tiles = [
            add_tile_label(working_rgb, title, tile_size),
            add_tile_label(fov.astype(np.uint8) * 255, 'FOV', tile_size),
            add_tile_label(gt_tile, 'GT', tile_size),
        ]
        for label, channel in debug_conditioning_candidates(working_rgb, fov, config):
            row_tiles.append(add_tile_label(channel, label, tile_size))
        sheet_rows.append(row_tiles)
    sheet = np.concatenate([np.concatenate(row_tiles, axis=1) for row_tiles in sheet_rows], axis=0)
    sheet_path = output_dir / 'debug_conditioning_visualization_15_samples.jpg'
    cv2.imwrite(str(sheet_path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    return {'sheet': sheet_path}


def local_std_float(channel: np.ndarray, fov: np.ndarray, kernel_size: int = 15) -> np.ndarray:
    filled = fill_outside_fov(channel, fov)
    kernel_size = max(3, int(kernel_size) | 1)
    mean = cv2.blur(filled, (kernel_size, kernel_size), borderType=cv2.BORDER_REFLECT)
    mean_sq = cv2.blur(filled * filled, (kernel_size, kernel_size), borderType=cv2.BORDER_REFLECT)
    std = np.sqrt(np.maximum(mean_sq - mean * mean, 0.0)).astype(np.float32)
    std[~fov] = 0
    return std


def high_frequency_energy(channel: np.ndarray, fov: np.ndarray, sigma: float = 2.0, kernel_size: int = 15) -> np.ndarray:
    filled = fill_outside_fov(channel, fov)
    smooth = cv2.GaussianBlur(filled, (0, 0), sigmaX=float(sigma), sigmaY=float(sigma), borderType=cv2.BORDER_REFLECT)
    high = np.abs(filled - smooth)
    energy = cv2.blur(high, (int(kernel_size) | 1, int(kernel_size) | 1), borderType=cv2.BORDER_REFLECT)
    energy[~fov] = 0
    return energy.astype(np.float32)


def coherence_gate(channel: np.ndarray, fov: np.ndarray, sigma: float = 1.2) -> np.ndarray:
    filled = fill_outside_fov(channel, fov).astype(np.float32) / 255.0
    gx = cv2.Sobel(filled, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(filled, cv2.CV_32F, 0, 1, ksize=3)
    jxx = cv2.GaussianBlur(gx * gx, (0, 0), sigmaX=float(sigma), sigmaY=float(sigma), borderType=cv2.BORDER_REFLECT)
    jyy = cv2.GaussianBlur(gy * gy, (0, 0), sigmaX=float(sigma), sigmaY=float(sigma), borderType=cv2.BORDER_REFLECT)
    jxy = cv2.GaussianBlur(gx * gy, (0, 0), sigmaX=float(sigma), sigmaY=float(sigma), borderType=cv2.BORDER_REFLECT)
    trace = jxx + jyy
    delta = np.sqrt(np.maximum((jxx - jyy) ** 2 + 4.0 * jxy ** 2, 0.0))
    gate = delta / (trace + 1e-6)
    gate[~fov] = 0
    return np.clip(gate, 0.0, 1.0).astype(np.float32)


def inverse_percentile_gate(values: np.ndarray, fov: np.ndarray, low_pct: float = 45.0, high_pct: float = 90.0) -> np.ndarray:
    if not np.any(fov):
        return np.zeros(values.shape, dtype=np.float32)
    low, high = np.percentile(values[fov], [float(low_pct), float(high_pct)])
    if high <= low:
        return fov.astype(np.float32)
    gate = 1.0 - np.clip((values.astype(np.float32) - float(low)) / float(high - low), 0.0, 1.0)
    gate[~fov] = 0
    return gate.astype(np.float32)


def capped_dark_detail(source: np.ndarray, fov: np.ndarray, sigma: float = 18.0, cap: float = 38.0) -> np.ndarray:
    filled = fill_outside_fov(source, fov)
    background = cv2.GaussianBlur(filled, (0, 0), sigmaX=float(sigma), sigmaY=float(sigma), borderType=cv2.BORDER_REFLECT)
    detail = np.maximum(background - filled, 0.0)
    detail = np.minimum(detail, float(cap))
    detail[~fov] = 0
    return detail.astype(np.float32)


def suppress_detail_artifacts(detail: np.ndarray, rgb: np.ndarray, fov: np.ndarray, rim_px: int = 18) -> np.ndarray:
    output = detail.astype(np.float32).copy()
    inner = erode_mask(fov, int(rim_px))
    output[fov & ~inner] = 0
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l_channel = lab[:, :, 0]
    if np.any(fov):
        bright = (l_channel >= np.percentile(l_channel[fov], 98.0)) & fov
        bright = cv2.dilate(bright.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (19, 19)), iterations=1).astype(bool)
        output[bright] = 0
    output[~fov] = 0
    return output.astype(np.float32)


def inject_dark_detail(base: np.ndarray, detail: np.ndarray, gate: np.ndarray, fov: np.ndarray, strength: float = 1.0) -> np.ndarray:
    conditioned = base.astype(np.float32) - float(strength) * detail.astype(np.float32) * gate.astype(np.float32)
    conditioned[~fov] = 0
    return np.clip(conditioned, 0, 255).astype(np.uint8)


def debug_conditional_detail_candidates(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> list[tuple[str, np.ndarray]]:
    c6 = cielab_green_clahe_source(rgb, fov, config, l_clip=6.0, green_clip=6.0)
    c4 = cielab_green_clahe_source(rgb, fov, config, l_clip=4.0, green_clip=4.0)
    detail = capped_dark_detail(c6, fov, sigma=18.0, cap=38.0)
    detail_artifact_safe = suppress_detail_artifacts(detail, rgb, fov, rim_px=18)
    std_gate = inverse_percentile_gate(local_std_float(c6, fov, kernel_size=15), fov, low_pct=45.0, high_pct=90.0)
    hf_gate = inverse_percentile_gate(high_frequency_energy(c6, fov, sigma=2.0, kernel_size=15), fov, low_pct=45.0, high_pct=90.0)
    coh_gate = coherence_gate(c6, fov, sigma=1.2)
    quiet_line_gate = np.clip(std_gate * (0.35 + 0.65 * coh_gate), 0.0, 1.0)
    hf_line_gate = np.clip(hf_gate * (0.35 + 0.65 * coh_gate), 0.0, 1.0)
    artifact_line_gate = np.clip(quiet_line_gate * (detail_artifact_safe > 0).astype(np.float32), 0.0, 1.0)
    combined_gate = np.clip(0.55 * quiet_line_gate + 0.30 * hf_line_gate + 0.15 * coh_gate, 0.0, 1.0)
    combined_detail = detail_artifact_safe
    return [
        ('C4G4 base', c4),
        ('C6G6 source', c6),
        ('C6 dark detail', fixed_positive_uint8(detail, scale=4.0)),
        ('Std quiet gate', fixed_positive_uint8(std_gate, scale=255.0)),
        ('HF quiet gate', fixed_positive_uint8(hf_gate, scale=255.0)),
        ('Coherence gate', fixed_positive_uint8(coh_gate, scale=255.0)),
        ('C4 + capped detail', inject_dark_detail(c4, detail, fov.astype(np.float32), fov, strength=0.85)),
        ('C4 + std gated detail', inject_dark_detail(c4, detail, std_gate, fov, strength=1.10)),
        ('C4 + HF gated detail', inject_dark_detail(c4, detail, hf_gate, fov, strength=1.10)),
        ('C4 + coherence detail', inject_dark_detail(c4, detail, coh_gate, fov, strength=0.90)),
        ('C4 + std line detail', inject_dark_detail(c4, detail, quiet_line_gate, fov, strength=1.20)),
        ('C4 + HF line detail', inject_dark_detail(c4, detail, hf_line_gate, fov, strength=1.20)),
        ('C4 + artifact safe detail', inject_dark_detail(c4, detail_artifact_safe, artifact_line_gate, fov, strength=1.20)),
        ('Combined conditional', inject_dark_detail(c4, combined_detail, combined_gate, fov, strength=1.15)),
    ]


def run_debug_conditional_detail_visualization(rows: List[Dict[str, object]], output_dir: Path, tile_size: int = 150) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    config = build_vessel_config()
    sheet_rows = []
    for idx, row in enumerate(tqdm(rows, desc='DEBUG conditional detail visualization'), start=1):
        rgb = read_rgb(row['image_path'])
        working_rgb = resize_max_side(rgb, config.process_max_side)
        fov = estimate_fov_mask(working_rgb)
        gt = read_binary_mask(row['mask_path'], fov.shape) if row.get('mask_path') is not None else None
        gt_tile = gt.astype(np.uint8) * 255 if gt is not None else empty_tile(fov.shape)
        title = f'{idx}. {row["source"]} {row["name"]}'
        row_tiles = [
            add_tile_label(working_rgb, title, tile_size),
            add_tile_label(fov.astype(np.uint8) * 255, 'FOV', tile_size),
            add_tile_label(gt_tile, 'GT', tile_size),
        ]
        for label, channel in debug_conditional_detail_candidates(working_rgb, fov, config):
            row_tiles.append(add_tile_label(channel, label, tile_size))
        sheet_rows.append(row_tiles)
    sheet = np.concatenate([np.concatenate(row_tiles, axis=1) for row_tiles in sheet_rows], axis=0)
    sheet_path = output_dir / 'debug_conditional_detail_visualization_15_samples.jpg'
    cv2.imwrite(str(sheet_path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    return {'sheet': sheet_path}


def artifact_safe_conditioned_source(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> np.ndarray:
    c6 = cielab_green_clahe_source(rgb, fov, config, l_clip=6.0, green_clip=6.0)
    c4 = cielab_green_clahe_source(rgb, fov, config, l_clip=4.0, green_clip=4.0)
    detail = capped_dark_detail(c6, fov, sigma=18.0, cap=38.0)
    detail_artifact_safe = suppress_detail_artifacts(detail, rgb, fov, rim_px=18)
    std_gate = inverse_percentile_gate(local_std_float(c6, fov, kernel_size=15), fov, low_pct=45.0, high_pct=90.0)
    coh_gate = coherence_gate(c6, fov, sigma=1.2)
    quiet_line_gate = np.clip(std_gate * (0.35 + 0.65 * coh_gate), 0.0, 1.0)
    artifact_line_gate = np.clip(quiet_line_gate * (detail_artifact_safe > 0).astype(np.float32), 0.0, 1.0)
    return inject_dark_detail(c4, detail_artifact_safe, artifact_line_gate, fov, strength=1.20)


def matched_filter_response_small(inverted_float: np.ndarray, fov: np.ndarray) -> np.ndarray:
    response = np.zeros(inverted_float.shape, dtype=np.float32)
    for size, sigma in ((5, 0.7), (7, 0.9), (9, 1.1), (11, 1.3)):
        for angle in range(0, 180, 10):
            kernel = line_kernel(size, angle, sigma)
            filtered = cv2.filter2D(inverted_float, cv2.CV_32F, kernel, borderType=cv2.BORDER_REFLECT)
            response = np.maximum(response, filtered)
    response[~fov] = 0
    return normalize01(response, fov)


def residual_from_source_channel(
    channel: np.ndarray,
    fov: np.ndarray,
    config: VesselPipelineConfig,
    apply_clahe: bool = True,
    clahe_clip: float | None = None,
) -> np.ndarray:
    source = np.clip(channel, 0, 255).astype(np.uint8)
    if apply_clahe:
        clip = float(config.clahe_clip if clahe_clip is None else clahe_clip)
        source = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8)).apply(source)
    source[~fov] = 0
    filtered = bilateral_filter_green(source, fov, config)
    background = estimate_background_mean(filtered, fov, int(round(config.background_sigma)))
    return normalize_green_with_background_residual(
        filtered,
        background,
        fov,
        stats_erode_px=int(config.fov_erode_px),
    )


def vessel_maps_from_enhanced_channel(
    enhanced_channel: np.ndarray,
    fov: np.ndarray,
    config: VesselPipelineConfig,
    include_small_branch: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    inverted = 255 - np.clip(enhanced_channel, 0, 255).astype(np.uint8)
    inverted_float = normalize01(inverted.astype(np.float32), fov)
    top_hat = modified_tophat(inverted, fov)
    matched = matched_filter_response(inverted_float, fov)
    frangi_map = normalize01(safe_filter_call('frangi', inverted_float), fov)
    jerman_map = jerman_vesselness(inverted_float, fov)
    combined = fuse_vessel_responses(
        top_hat,
        matched,
        frangi_map,
        jerman_map,
        fov,
        mode="almeida" if config.vesselness_mode == "almeida_paper" else config.vesselness_mode,
        config=config,
    )
    if include_small_branch:
        small_matched = matched_filter_response_small(inverted_float, fov)
        small_jerman = jerman_vesselness(inverted_float, fov, sigmas=(0.6, 0.8, 1.0, 1.2))
        combined = normalize01(0.72 * combined + 0.18 * small_matched + 0.10 * small_jerman, fov)
    p10 = threshold_response_map(
        combined,
        fov,
        method='percentile',
        target_density=0.10,
        fov_erode_px=max(0, int(config.fov_erode_px)),
    )
    return combined, p10


def vessel_filter_maps_from_enhanced_channel(
    enhanced_channel: np.ndarray,
    fov: np.ndarray,
    config: VesselPipelineConfig,
) -> list[tuple[str, np.ndarray]]:
    inverted = 255 - np.clip(enhanced_channel, 0, 255).astype(np.uint8)
    inverted[~fov] = 0
    inverted_float = normalize01(inverted.astype(np.float32), fov)
    top_hat = modified_tophat(inverted, fov)
    matched = matched_filter_response(inverted_float, fov)
    frangi_map = normalize01(safe_filter_call('frangi', inverted_float), fov)
    jerman_map = jerman_vesselness(inverted_float, fov)
    combined = fuse_vessel_responses(
        top_hat,
        matched,
        frangi_map,
        jerman_map,
        fov,
        mode="almeida" if config.vesselness_mode == "almeida_paper" else config.vesselness_mode,
        config=config,
    )
    return [
        ('source', enhanced_channel),
        ('inverted', inverted),
        ('top-hat', top_hat),
        ('matched', matched),
        ('frangi', frangi_map),
        ('jerman', jerman_map),
        ('combined', combined),
    ]


def conditioned_filter_preview_sources(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> list[tuple[str, np.ndarray]]:
    return [
        ('C4G4', cielab_green_clahe_source(rgb, fov, config, l_clip=4.0, green_clip=4.0)),
        ('C6G6', cielab_green_clahe_source(rgb, fov, config, l_clip=6.0, green_clip=6.0)),
        ('C4 artifact safe', artifact_safe_conditioned_source(rgb, fov, config)),
    ]


def run_debug_conditioned_source_filter_preview(rows: List[Dict[str, object]], output_dir: Path, tile_size: int = 150) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    config = build_vessel_config()
    sheet_rows = []

    for idx, row in enumerate(tqdm(rows, desc='DEBUG conditioned source filter preview'), start=1):
        rgb = read_rgb(row['image_path'])
        working_rgb = resize_max_side(rgb, config.process_max_side)
        fov = estimate_fov_mask(working_rgb)
        gt = read_binary_mask(row['mask_path'], fov.shape) if row.get('mask_path') is not None else None
        gt_tile = gt.astype(np.uint8) * 255 if gt is not None else empty_tile(fov.shape)
        title = f'{idx}. {row["source"]} {row["name"]}'
        row_tiles = [
            add_tile_label(working_rgb, title, tile_size),
            add_tile_label(fov.astype(np.uint8) * 255, 'FOV', tile_size),
            add_tile_label(gt_tile, 'GT', tile_size),
        ]

        for label, source_channel in conditioned_filter_preview_sources(working_rgb, fov, config):
            for map_label, filter_map in vessel_filter_maps_from_enhanced_channel(source_channel, fov, config):
                row_tiles.append(add_tile_label(filter_map, f'{label} {map_label}', tile_size))
        sheet_rows.append(row_tiles)

    sheet = np.concatenate([np.concatenate(row_tiles, axis=1) for row_tiles in sheet_rows], axis=0)
    sheet_path = output_dir / 'debug_conditioned_source_filter_maps_15_samples.jpg'
    cv2.imwrite(str(sheet_path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    return {'sheet': sheet_path}


def run_debug_flatten_darkline_filter_visualization(rows: List[Dict[str, object]], output_dir: Path, tile_size: int = 150) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    config = build_vessel_config()
    sheet_rows = []

    for idx, row in enumerate(tqdm(rows, desc='DEBUG flatten/dark-line filter maps'), start=1):
        rgb = read_rgb(row['image_path'])
        working_rgb = resize_max_side(rgb, config.process_max_side)
        fov = estimate_fov_mask(working_rgb)
        gt = read_binary_mask(row['mask_path'], fov.shape) if row.get('mask_path') is not None else None
        gt_tile = gt.astype(np.uint8) * 255 if gt is not None else empty_tile(fov.shape)
        title = f'{idx}. {row["source"]} {row["name"]}'
        row_tiles = [
            add_tile_label(working_rgb, title, tile_size),
            add_tile_label(fov.astype(np.uint8) * 255, 'FOV', tile_size),
            add_tile_label(gt_tile, 'GT', tile_size),
        ]

        for label, source_channel in debug_flatten_darkline_filter_sources(working_rgb, fov, config):
            for map_label, filter_map in vessel_filter_maps_from_enhanced_channel(source_channel, fov, config):
                row_tiles.append(add_tile_label(filter_map, f'{label} {map_label}', tile_size))
        sheet_rows.append(row_tiles)

    sheet = np.concatenate([np.concatenate(row_tiles, axis=1) for row_tiles in sheet_rows], axis=0)
    sheet_path = output_dir / 'debug_flatten_darkline_filter_maps_15_samples.jpg'
    cv2.imwrite(str(sheet_path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))
    return {'sheet': sheet_path}


def channel_source_candidates(rgb: np.ndarray, fov: np.ndarray, config: VesselPipelineConfig) -> list[tuple[str, np.ndarray, np.ndarray, bool]]:
    cielab_rgb, green_clahe_current, _, _, _, _, current_residual, _ = preprocess_green_channel(rgb, fov, config)
    cielab_green = cielab_rgb[:, :, 1].copy()
    cielab_green[~fov] = 0

    local16_source = local_clahe_channel(cielab_green, fov, clip_limit=2.2, tile_grid_size=(16, 16))
    old8_source = local_clahe_channel(cielab_green, fov, clip_limit=float(config.clahe_clip), tile_grid_size=(8, 8))
    blackhat_source = small_blackhat_boost(old8_source, fov, strength=0.75)

    old8_residual = residual_from_source_channel(old8_source, fov, config, apply_clahe=False)
    local16_residual = residual_from_source_channel(local16_source, fov, config, apply_clahe=False)
    blackhat_residual = residual_from_source_channel(blackhat_source, fov, config, apply_clahe=False)

    return [
        ('CIELAB green 8x8', old8_source, old8_residual, False),
        ('CIELAB green local16', green_clahe_current, current_residual, False),
        ('CIELAB blackhat', blackhat_source, blackhat_residual, False),
        ('CIELAB local16 small', green_clahe_current, current_residual, True),
    ]


def run_debug_channel_source_visualization(rows: List[Dict[str, object]], output_dir: Path, tile_size: int = 150) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    config = build_vessel_config()
    metric_rows = []
    sheet_rows = []

    for idx, row in enumerate(tqdm(rows, desc='DEBUG channel source visualization'), start=1):
        rgb = read_rgb(row['image_path'])
        working_rgb = resize_max_side(rgb, config.process_max_side)
        fov = estimate_fov_mask(working_rgb)
        gt = read_binary_mask(row['mask_path'], fov.shape) if row['mask_path'] is not None else None
        gt_tile = gt if gt is not None else empty_tile(fov.shape)
        title = f'{idx}. {row["source"]} {row["name"]}'
        row_tiles = [
            add_tile_label(working_rgb, title, tile_size),
            add_tile_label(fov, 'FOV', tile_size),
            add_tile_label(gt_tile, 'GT', tile_size),
        ]

        for label, source_channel, normalized, include_small_branch in channel_source_candidates(working_rgb, fov, config):
            combined, p10 = vessel_maps_from_enhanced_channel(
                normalized,
                fov,
                config,
                include_small_branch=include_small_branch,
            )
            row_tiles.extend([
                add_tile_label(source_channel, f'{label} source', tile_size),
                add_tile_label(combined, f'{label} comb', tile_size),
                add_tile_label(p10, f'{label} P10', tile_size),
            ])
            if gt is not None:
                metrics = {
                    **debug_binary_metrics(p10 > 0, gt),
                    **debug_soft_metrics(p10, gt),
                }
                metric_rows.append({
                    'source': row['source'],
                    'image': row['name'],
                    'channel_source': label,
                    **metrics,
                })
        sheet_rows.append(row_tiles)

    sheet = np.concatenate([np.concatenate(row_tiles, axis=1) for row_tiles in sheet_rows], axis=0)
    sheet_path = output_dir / 'debug_channel_source_15_samples.jpg'
    cv2.imwrite(str(sheet_path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))

    metrics_path = output_dir / 'debug_channel_source_metrics.csv'
    write_rows_csv(metrics_path, metric_rows)

    summary_rows = []
    if metric_rows:
        for channel_source in sorted({row['channel_source'] for row in metric_rows}):
            selected = [row for row in metric_rows if row['channel_source'] == channel_source]
            numeric_keys = [key for key in selected[0].keys() if key not in {'source', 'image', 'channel_source'}]
            summary_rows.append({
                'channel_source': channel_source,
                **{key: float(np.mean([float(row[key]) for row in selected])) for key in numeric_keys},
            })
    summary_path = output_dir / 'debug_channel_source_summary.csv'
    write_rows_csv(summary_path, summary_rows)
    return {'sheet': sheet_path, 'metrics': metrics_path, 'summary': summary_path}


def run_debug_baseline_visualization(output_dir: Path, tile_size: int = 150) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    config = build_vessel_config()

    rows = []
    rows.extend(find_agrawal_debug_subset(AGRAWAL_ROOT, 'RetCam', 5))
    rows.extend(find_agrawal_debug_subset(AGRAWAL_ROOT, 'Neo', 5))
    rows.extend(find_zhao_debug_samples(ZHAO_ROOT, 5))
    if len(rows) != 15:
        raise RuntimeError(f'Expected 15 debug rows, found {len(rows)}')

    tuning_path = run_vessel_fusion_tuning(rows, output_dir)
    channel_paths = run_debug_channel_source_visualization(rows, output_dir, tile_size=tile_size)

    metric_rows = []
    threshold_variant_metric_rows = []
    sheet_rows = []
    for idx, row in enumerate(tqdm(rows, desc='DEBUG baseline visualization'), start=1):
        rgb = read_rgb(row['image_path'])
        debug = compute_vessel_debug_maps(rgb, config)
        threshold_variants = threshold_debug_variants(
            debug.combined,
            debug.fov,
            fov_erode_px=max(0, int(config.fov_erode_px)),
        )
        gt = read_binary_mask(row['mask_path'], debug.fov.shape) if row['mask_path'] is not None else None
        tri_pred = debug.triangle_clean > 0
        final_pred = debug.final_binary

        if gt is not None:
            for variant_label, variant_map in threshold_variants:
                variant_pred = variant_map > 0
                variant_metrics = {
                    **debug_binary_metrics(variant_pred, gt),
                    **debug_soft_metrics(variant_map, gt),
                }
                threshold_variant_metric_rows.append({
                    'source': row['source'],
                    'image': row['name'],
                    'variant': variant_label,
                    **variant_metrics,
                })
            tri_metrics = {
                f'threshold_clean_{key}': value
                for key, value in {**debug_binary_metrics(tri_pred, gt), **debug_soft_metrics(debug.triangle_clean, gt)}.items()
            }
            final_metrics = {f'final_binary_{key}': value for key, value in debug_binary_metrics(final_pred, gt).items()}
            metric_rows.append({'source': row['source'], 'image': row['name'], **tri_metrics, **final_metrics})
            gt_tile = gt
            overlay = overlay_prediction(final_pred, gt)
        else:
            gt_tile = empty_tile(debug.fov.shape)
            overlay = empty_tile(debug.fov.shape)

        title = f'{idx}. {row["source"]} {row["name"]}'
        sheet_rows.append([
            add_tile_label(debug.raw_rgb, title, tile_size),
            add_tile_label(debug.fov, 'FOV', tile_size),
            add_tile_label(debug.cielab_rgb, 'CIELAB', tile_size),
            add_tile_label(debug.green, 'Green CLAHE', tile_size),
            add_tile_label(debug.green_filtered, 'Bilateral green', tile_size),
            add_tile_label(debug.background_new, 'Background', tile_size),
            add_tile_label(debug.normalized_fixed, 'Fixed norm', tile_size),
            add_tile_label(debug.normalized_new, 'Residual norm', tile_size),
            add_tile_label(debug.normalized_clamped, 'Clamped norm', tile_size),
            add_tile_label(debug.normalized_light_residual, 'Light residual', tile_size),
            add_tile_label(debug.normalized_weak_residual, 'Weak residual', tile_size),
            add_tile_label(debug.normalized_light_weak_residual, 'Light weak', tile_size),
            add_tile_label(debug.normalized_clahe3_weak_residual, 'CLAHE3 weak', tile_size),
            add_tile_label(debug.bsgmf, 'BSGMF', tile_size),
            add_tile_label(debug.top_hat, 'Top-hat', tile_size),
            add_tile_label(debug.frangi, 'Frangi', tile_size),
            add_tile_label(debug.hessian, 'Hessian', tile_size),
            add_tile_label(debug.combined, 'Combined', tile_size),
            *[
                add_tile_label(variant_map, variant_label, tile_size)
                for variant_label, variant_map in threshold_variants
            ],
            add_tile_label(debug.triangle_clean, 'Active clean', tile_size),
            add_tile_label(gt_tile, 'GT', tile_size),
            add_tile_label(overlay, 'Overlay', tile_size),
            add_tile_label(final_pred, 'Final binary', tile_size),
        ])

    sheet = np.concatenate([np.concatenate(row_tiles, axis=1) for row_tiles in sheet_rows], axis=0)
    sheet_path = output_dir / 'debug_baseline_15_samples.jpg'
    cv2.imwrite(str(sheet_path), cv2.cvtColor(sheet, cv2.COLOR_RGB2BGR))

    metrics_path = output_dir / 'debug_agrawal_sample_metrics.csv'
    write_rows_csv(metrics_path, metric_rows)

    threshold_variant_metrics_path = output_dir / 'debug_threshold_variant_metrics.csv'
    write_rows_csv(threshold_variant_metrics_path, threshold_variant_metric_rows)

    threshold_variant_summary_rows = []
    if threshold_variant_metric_rows:
        variant_names = sorted({row['variant'] for row in threshold_variant_metric_rows})
        for variant_name in variant_names:
            selected = [row for row in threshold_variant_metric_rows if row['variant'] == variant_name]
            numeric_keys = [key for key in selected[0].keys() if key not in {'source', 'image', 'variant'}]
            threshold_variant_summary_rows.append({
                'variant': variant_name,
                **{key: float(np.mean([float(row[key]) for row in selected])) for key in numeric_keys},
            })
    threshold_variant_summary_path = output_dir / 'debug_threshold_variant_summary.csv'
    write_rows_csv(threshold_variant_summary_path, threshold_variant_summary_rows)

    summary_rows = []
    if metric_rows:
        for source in ('RetCam', 'Neo', 'All'):
            selected = metric_rows if source == 'All' else [row for row in metric_rows if row['source'] == source]
            if not selected:
                continue
            numeric_keys = [key for key in selected[0].keys() if key not in {'source', 'image'}]
            summary_rows.append({
                'source': source,
                **{key: float(np.mean([float(row[key]) for row in selected])) for key in numeric_keys},
            })
    summary_path = output_dir / 'debug_agrawal_sample_summary.csv'
    write_rows_csv(summary_path, summary_rows)

    config_path = output_dir / 'debug_baseline_config.csv'
    write_rows_csv(config_path, [asdict(config)])

    print(f'Saved DEBUG sheet: {sheet_path}')
    print(f'Saved DEBUG metrics: {metrics_path}')
    print(f'Saved DEBUG threshold variant metrics: {threshold_variant_metrics_path}')
    print(f'Saved DEBUG threshold variant summary: {threshold_variant_summary_path}')
    print(f'Saved DEBUG summary: {summary_path}')
    print(f'Saved DEBUG tuning summary: {tuning_path}')
    print(f'Saved DEBUG channel source sheet: {channel_paths['sheet']}')
    print(f'Saved DEBUG channel source summary: {channel_paths['summary']}')
    return {
        'sheet': sheet_path,
        'metrics': metrics_path,
        'threshold_variant_metrics': threshold_variant_metrics_path,
        'threshold_variant_summary': threshold_variant_summary_path,
        'summary': summary_path,
        'config': config_path,
        'tuning': tuning_path,
        'channel_source_sheet': channel_paths['sheet'],
        'channel_source_metrics': channel_paths['metrics'],
        'channel_source_summary': channel_paths['summary'],
    }


if cfg.debug_mode:
    debug_paths = run_debug_baseline_visualization(Path(cfg.debug_output_dir), tile_size=cfg.debug_tile_size)
    debug_sheet = cv2.cvtColor(cv2.imread(str(debug_paths['sheet'])), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(32, 24))
    plt.imshow(debug_sheet)
    plt.axis('off')
    plt.show()
else:
    debug_paths = {}
    print('DEBUG MODE is disabled. Set cfg.debug_mode = True to regenerate the 15-image baseline visualization.')

## Scenario Cache Generation

The generated scenario images are saved as PNG files. The cache is deterministic and can be reused across training runs.

In [ ]:
PROCESSED_DIR = OUTPUT_PROCESSED_DIR


def cache_path_for(row: pd.Series, scenario: str) -> Path:
    source = Path(row['path'])
    return PROCESSED_DIR / scenario / row['label'] / f'{source.stem}.png'


def generate_scenario_images_for_row(row: pd.Series) -> Dict[str, np.ndarray]:
    original = read_rgb(row['path'])
    raw = resize_rgb(original, cfg.image_size)
    enhanced = enhance_for_classification(original)
    soft, binary, fov = segment_vessels(original)
    soft_224 = resize_soft_map(soft, cfg.image_size)
    binary_224 = resize_binary_mask(binary, cfg.image_size)
    fov_224 = resize_binary_mask(fov, cfg.image_size)
    vessel_mask = np.repeat((soft_224 * 255).astype(np.uint8)[:, :, None], 3, axis=2)
    guided = vessel_guided_image(enhanced, soft_224, fov_224)
    return {
        'S1_raw': raw,
        'S2_enhanced': enhanced,
        'S3_vessel_mask': vessel_mask,
        'S4_vessel_guided': guided,
    }


def ensure_scenario_cache(df: pd.DataFrame, scenarios: Tuple[str, ...] = SCENARIOS, force: bool = False) -> pd.DataFrame:
    df = df.copy()
    for scenario in scenarios:
        df[f'{scenario}_path'] = [str(cache_path_for(row, scenario)) for _, row in df.iterrows()]

    missing_rows = []
    for idx, row in df.iterrows():
        expected_paths = [cache_path_for(row, scenario) for scenario in scenarios]
        if force or any(not path.exists() for path in expected_paths):
            missing_rows.append(idx)

    print(f'Rows requiring preprocessing: {len(missing_rows)} / {len(df)}')
    for idx in tqdm(missing_rows, desc='Generating scenario cache'):
        row = df.loc[idx]
        generated = generate_scenario_images_for_row(row)
        for scenario in scenarios:
            write_rgb(cache_path_for(row, scenario), generated[scenario])

    cached_csv = OUTPUT_SPLITS_DIR / 'zhao_split_with_cache_paths.csv'
    df.to_csv(cached_csv, index=False)
    print(f'Saved cache-aware split file: {cached_csv}')
    return df

if cfg.run_preprocessing:
    split_df = ensure_scenario_cache(split_df, SCENARIOS, force=cfg.force_rebuild_cache)
else:
    cached_csv = OUTPUT_SPLITS_DIR / 'zhao_split_with_cache_paths.csv'
    if cached_csv.exists():
        split_df = pd.read_csv(cached_csv)
    else:
        print('Preprocessing is disabled and no cache-aware split file exists yet.')

In [ ]:
def show_scenario_check(df: pd.DataFrame, n: int = 3) -> None:
    available = all(f'{scenario}_path' in df.columns for scenario in SCENARIOS)
    if not available:
        print('Scenario cache paths are not available yet.')
        return
    sample = df.groupby('label', group_keys=False).sample(n=1, random_state=cfg.seed).head(n)
    for row in sample.itertuples(index=False):
        images = [read_rgb(getattr(row, f'{scenario}_path')) for scenario in SCENARIOS]
        show_image_grid(images, [f'{scenario} | {row.label}' for scenario in SCENARIOS], cols=4, figsize=(14, 4))

show_scenario_check(split_df, n=4)

## PyTorch Dataset and Dataloader

All scenarios are loaded as 3-channel images. S3 stores the binary vessel mask repeated into RGB channels, so the same Tiny ResNet architecture can be used for S1-S4.

In [ ]:
def build_transform(split: str, scenario: str):
    if A is None:
        return None
    transforms = []
    if split == 'train':
        transforms.extend([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.Rotate(limit=12, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
        ])
        if scenario != 'S3_vessel_mask':
            transforms.append(A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.35))
    transforms.append(A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)))
    return A.Compose(transforms)


class ROPScenarioDataset(Dataset):
    def __init__(self, df: pd.DataFrame, scenario: str, split: str):
        self.df = df[df['split'] == split].reset_index(drop=True)
        self.scenario = scenario
        self.split = split
        self.transform = build_transform(split, scenario)
        self.path_col = f'{scenario}_path'
        if self.path_col not in self.df.columns:
            raise ValueError(f'Missing cache path column: {self.path_col}')

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image = read_rgb(row[self.path_col])
        image = resize_rgb(image, cfg.image_size)
        if self.transform is not None:
            image = self.transform(image=image)['image']
        else:
            image = image.astype(np.float32) / 255.0
            image = (image - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
        image = torch.from_numpy(image.transpose(2, 0, 1)).float()
        label = torch.tensor(int(row['label_id']), dtype=torch.long)
        return image, label


def make_dataloaders(scenario: str, batch_size: int) -> Dict[str, DataLoader]:
    loaders = {}
    for split in ['train', 'val', 'test']:
        dataset = ROPScenarioDataset(split_df, scenario, split)
        shuffle = split == 'train'
        loaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=cfg.num_workers,
            pin_memory=(DEVICE.type == 'cuda'),
            persistent_workers=(cfg.num_workers > 0),
        )
    return loaders

## Models

Tiny ResNet is the main model for all four scenarios. ResNet50 is optional and only intended for S1 and S2 because those are ordinary RGB image scenarios and Zhao2024 reported ResNet50 as a strong baseline.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out)


class TinyResNet(nn.Module):
    def __init__(self, num_classes: int = 4, widths: Tuple[int, int, int] = (32, 64, 128)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, widths[0], kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(widths[0]),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.layer1 = self._make_layer(widths[0], widths[0], blocks=2, stride=1)
        self.layer2 = self._make_layer(widths[0], widths[1], blocks=2, stride=2)
        self.layer3 = self._make_layer(widths[1], widths[2], blocks=2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.25)
        self.fc = nn.Linear(widths[2], num_classes)

    @staticmethod
    def _make_layer(in_channels: int, out_channels: int, blocks: int, stride: int):
        layers = [BasicBlock(in_channels, out_channels, stride=stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)


def build_tiny_resnet() -> nn.Module:
    return TinyResNet(num_classes=len(cfg.class_names))


def build_resnet50(pretrained: bool = False) -> nn.Module:
    weights = None
    if pretrained:
        try:
            weights = models.ResNet50_Weights.DEFAULT
        except Exception:
            weights = None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, len(cfg.class_names))
    return model


def prepare_model(model: nn.Module) -> nn.Module:
    model = model.to(DEVICE)
    if USE_MULTI_GPU:
        model = nn.DataParallel(model)
    return model


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model

In [ ]:
def class_weights_for_training(df: pd.DataFrame) -> torch.Tensor:
    train_counts = df[df['split'] == 'train']['label_id'].value_counts().sort_index()
    counts = np.array([train_counts.get(i, 0) for i in range(len(cfg.class_names))], dtype=np.float32)
    weights = counts.sum() / (len(counts) * np.maximum(counts, 1.0))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def metric_summary(y_true: List[int], y_pred: List[int]) -> Dict[str, float]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(len(cfg.class_names))),
        average='macro',
        zero_division=0,
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
    }


def run_epoch(model, loader, criterion, optimizer=None, scaler=None) -> Tuple[float, Dict[str, float], List[int], List[int]]:
    is_train = optimizer is not None
    model.train(is_train)
    losses = []
    y_true, y_pred = [], []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(images)
                loss = criterion(logits, labels)
            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        losses.append(float(loss.detach().cpu()))
        preds = torch.argmax(logits.detach(), dim=1)
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

    return float(np.mean(losses)), metric_summary(y_true, y_pred), y_true, y_pred


def save_checkpoint(path: Path, model: nn.Module, metadata: Dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({'state_dict': unwrap_model(model).state_dict(), 'metadata': metadata}, path)


def load_checkpoint(path: Path, model: nn.Module) -> nn.Module:
    checkpoint = torch.load(path, map_location=DEVICE)
    unwrap_model(model).load_state_dict(checkpoint['state_dict'])
    return model

In [ ]:
def train_experiment(scenario: str, model_name: str = 'tinyresnet') -> Tuple[pd.DataFrame, Dict[str, float], np.ndarray]:
    seed_everything(cfg.seed)
    batch_size = cfg.batch_size_resnet50 if model_name == 'resnet50' else cfg.batch_size
    loaders = make_dataloaders(scenario, batch_size=batch_size)

    if model_name == 'tinyresnet':
        model = build_tiny_resnet()
    elif model_name == 'resnet50':
        model = build_resnet50(pretrained=False)
    else:
        raise ValueError(f'Unknown model_name: {model_name}')

    model = prepare_model(model)
    criterion = nn.CrossEntropyLoss(weight=class_weights_for_training(split_df))
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_f1 = -1.0
    best_epoch = -1
    stale_epochs = 0
    history = []
    checkpoint_path = OUTPUT_MODELS_DIR / f'{model_name}_{scenario}.pt'

    for epoch in range(1, cfg.epochs + 1):
        start = time.time()
        train_loss, train_metrics, _, _ = run_epoch(model, loaders['train'], criterion, optimizer=optimizer, scaler=scaler)
        val_loss, val_metrics, _, _ = run_epoch(model, loaders['val'], criterion)
        scheduler.step()

        row = {
            'epoch': epoch,
            'scenario': scenario,
            'model': model_name,
            'train_loss': train_loss,
            'val_loss': val_loss,
            **{f'train_{k}': v for k, v in train_metrics.items()},
            **{f'val_{k}': v for k, v in val_metrics.items()},
            'seconds': time.time() - start,
        }
        history.append(row)
        print(f"[{model_name} | {scenario}] epoch {epoch:03d} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_macro_f1={val_metrics['macro_f1']:.4f}")

        if val_metrics['macro_f1'] > best_f1:
            best_f1 = val_metrics['macro_f1']
            best_epoch = epoch
            stale_epochs = 0
            save_checkpoint(checkpoint_path, model, {'scenario': scenario, 'model': model_name, 'best_epoch': best_epoch, 'best_val_macro_f1': best_f1, 'cfg': asdict(cfg)})
        else:
            stale_epochs += 1
            if stale_epochs >= cfg.patience:
                print(f'Early stopping at epoch {epoch}. Best epoch: {best_epoch}')
                break

    history_df = pd.DataFrame(history)
    history_path = OUTPUT_RESULTS_DIR / f'history_{model_name}_{scenario}.csv'
    history_df.to_csv(history_path, index=False)

    model = load_checkpoint(checkpoint_path, model)
    test_loss, test_metrics, y_true, y_pred = run_epoch(model, loaders['test'], criterion)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(cfg.class_names))))
    test_metrics.update({'test_loss': test_loss, 'best_epoch': best_epoch, 'best_val_macro_f1': best_f1})

    report = classification_report(y_true, y_pred, target_names=list(cfg.class_names), zero_division=0)
    print(report)
    return history_df, test_metrics, cm


def plot_confusion_matrix(cm: np.ndarray, title: str) -> None:
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cfg.class_names, yticklabels=cfg.class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    plt.tight_layout()
    plt.show()

## Train Tiny ResNet for S1-S4

Set `cfg.run_training = True` in the configuration cell before running this section. On Kaggle 2x T4, the notebook uses AMP and `DataParallel` when both GPUs are visible.

In [ ]:
all_results = []
confusion_matrices = {}

if cfg.run_training:
    for scenario in SCENARIOS:
        history_df, test_metrics, cm = train_experiment(scenario, model_name='tinyresnet')
        result = {'model': 'tinyresnet', 'scenario': scenario, **test_metrics}
        all_results.append(result)
        confusion_matrices[('tinyresnet', scenario)] = cm
        plot_confusion_matrix(cm, f'Tiny ResNet - {scenario}')
else:
    print('Training is disabled. Set cfg.run_training = True in the configuration cell to train Tiny ResNet models.')

## Optional ResNet50 Baseline for S1 and S2

This baseline is limited to S1 and S2. It is not used for S3/S4 because our main comparison should focus on the image processing scenarios with one consistent custom CNN.

In [ ]:
if cfg.run_training and cfg.run_resnet50_baseline:
    for scenario in ['S1_raw', 'S2_enhanced']:
        history_df, test_metrics, cm = train_experiment(scenario, model_name='resnet50')
        result = {'model': 'resnet50', 'scenario': scenario, **test_metrics}
        all_results.append(result)
        confusion_matrices[('resnet50', scenario)] = cm
        plot_confusion_matrix(cm, f'ResNet50 - {scenario}')
else:
    print('ResNet50 baseline is disabled. Set cfg.run_training=True and cfg.run_resnet50_baseline=True to run it.')

## Final Comparison

The main model selection metric is test macro F1 after selecting the best checkpoint by validation macro F1. Accuracy is still reported, but macro F1 is more appropriate because Stage 1 is the minority class.

In [ ]:
results_path = OUTPUT_RESULTS_DIR / 'classification_comparison.csv'

if all_results:
    results_df = pd.DataFrame(all_results)
    results_df = results_df.sort_values(['model', 'scenario']).reset_index(drop=True)
    results_df.to_csv(results_path, index=False)
    display(results_df)
    print(f'Saved classification comparison: {results_path}')

    plt.figure(figsize=(10, 5))
    sns.barplot(data=results_df, x='scenario', y='macro_f1', hue='model')
    plt.ylim(0, 1)
    plt.xticks(rotation=20)
    plt.title('Test Macro F1 by Scenario')
    plt.tight_layout()
    figure_path = OUTPUT_FIGURES_DIR / 'test_macro_f1_by_scenario.png'
    plt.savefig(figure_path, dpi=150)
    plt.show()
    print(f'Saved figure: {figure_path}')
else:
    print('No training results yet.')

In [ ]:
summary = {
    'zhao_root': str(ZHAO_ROOT),
    'agrawal_root': str(AGRAWAL_ROOT) if AGRAWAL_ROOT is not None else None,
    'is_kaggle': IS_KAGGLE,
    'device': str(DEVICE),
    'gpu_count': GPU_COUNT,
    'scenarios': list(SCENARIOS),
    'class_names': list(cfg.class_names),
    'split_counts': pd.crosstab(split_df['split'], split_df['label']).to_dict(),
    'notes': [
        'Split is stratified by class but image-level because patient/eye identifiers are not available in local filenames.',
        'S3 uses cleaned binary vessel masks repeated into three channels.',
        'S4 uses cleaned binary masks converted to blurred soft guides and multiplied with enhanced RGB images.',
        'Deterministic preprocessing is cached; random augmentation is applied during training only.',
    ],
}
summary_path = OUTPUT_RESULTS_DIR / 'workflow_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print(f'Saved workflow summary: {summary_path}')